# YOLO26m · combo 단일 holdout NMS-free(one2one 헤드) + AI Hub 외부 데이터 (CUDA·MPS)

`exp_yolo26m_nms_free.ipynb`를 복사해 **train에만 AI Hub 외부 사진을 더한** 실험입니다.
외부 사진은 `02_eda.ipynb` §5~8과 같은 방법·seed로 고르고, 팀원 yolo11m AI Hub 노트북
(`yolo11m_100_aihub_not_background.ipynb`)과 같이 §8 문제 사진을 뺍니다(2-2절).
validation·평가·속도·제출 csv는 **대회 사진만** 씁니다.
`USE_AIHUB = False`로 바꾸면 원본 NMS-free 노트북과 같은 데이터로 학습합니다.

프로젝트의 **uv Python 3.12 가상환경을 커널로 선택**한 뒤 위에서 아래로 실행합니다.
데이터 준비 셀은 준비 파일을 만들고, **학습 셀은 단일 train/validation으로 한 번 학습**합니다.
장치는 `get_device()`로 CUDA > MPS 순으로 고르며 **CUDA·MPS 모두 본 학습**을 할 수 있습니다.
동작 확인만 하려면 설정 셀에서 `SMOKE = True`(1 epoch, 학습 데이터 10%)로 바꿉니다.
이미 학습한 실험은 설정 셀의 `REUSE_DIR`에 그 결과 폴더를 넣으면 학습을 건너뛰고
그 best.pt로 평가·그림·속도·제출 csv만 다시 만듭니다. 학습이 중간에 끊긴 실험이면 이어서 학습합니다.
작성된 노트북에는 실행 결과를 저장하지 않았습니다.

**Colab에서 실행하기** (0절이 Colab에서만 준비합니다. Mac에서는 0절이 아무것도 하지 않습니다)

1. 이 브랜치(`exp/yolo26m_nms_free_aihub`)의 노트북을 Colab에서 열고 **런타임 → 런타임 유형 변경**에서
   A100 또는 L4 GPU를 고릅니다.
2. 위에서 아래로 실행합니다. 0절이 Drive 마운트, 저장소 받기, 패키지 설치, 대회 데이터 받기를 합니다.
   대회 데이터 링크(`DATA_URL`)와 AI Hub 링크(`AIHUB_URL`)는 입력창이 뜨면 붙여 넣습니다(화면에 표시되지 않음).
3. 결과 폴더는 Drive의 `MyDrive/pillbbox_runs/<실험 이름>`이라 런타임이 끊겨도 `best.pt`·`last.pt`가 남습니다.
   학습 입력(`prepared`)과 외부 데이터는 런타임 디스크에 둡니다(약 48GB, 받고 푸는 데 수십 분).
4. **끊기면 이어 학습**: 새 런타임에서 설정 셀의 `REUSE_DIR`에 그 실험 폴더 이름을 넣고 처음부터 다시 실행합니다.
   데이터는 같은 seed로 다시 만들고(이전과 같은지 검사), 학습은 `train/weights/last.pt`에서 이어 갑니다.
   조기 중단 카운트(최고 점수와 그 에폭)도 이어서 셉니다.

| 항목 | 설정 |
|---|---|
| 모델 / 전이학습 | yolo26m.pt / full fine tuning (freeze=None) |
| 원본 | train 232장, test 842장. test는 학습·검증에서 제외 |
| 외부 데이터 | AI Hub 경구약제 조합(TL_2·TS_2 제외). 공통 클래스 전부 157 bbox까지 보충, 선택 bbox가 든 사진을 통째로 **train에만** 추가 |
| 외부 라벨 보정 | 적용하지 않음. 02_eda §10~11 OpenCV 보정은 사람 검수 전이라 쓰지 않음(yolo11m과 같음) |
| 외부 전용 알약 | 대회 56종에 없는 약은 **새 클래스**로 학습하고 평가·제출 예측에서는 뺀다 |
| 외부 사진 제외 | 대회 사진과 같은 이름(02_eda §6), 02_eda §8 문제 사진(라벨 누락·파일명에 없는 라벨·bbox 겹침·좌표 오류·JSON과 실제 크기 다름), 대회 사진과 같은 약 조합 |
| 라벨 | 현재 파이프라인의 v1 보정 라벨. 원본 JSON을 수정하지 않음 |
| 분할 | combo 단위 단일 holdout, 희귀 조합 train 고정, 교차검증 없음, seed=42 |
| 희귀 알약 | v1 전체 bbox가 5개 이하인 클래스가 든 **조합 전체를 train에 고정** |
| 비율 | 고정분을 제외한 조합을 약 8:2로 분할. 전체 이미지를 185/47로 강제하지 않음 |
| epochs / patience | 최대 100 / mAP75–95가 10에폭 연속 개선되지 않으면 중단 |
| optimizer / lr0 | MuSGD / 0.001 |
| batch / imgsz | 16 / 640 |
| 증강 | 기존 AUG_PRESETS["on"]: 회전 우선, 반전 끔, 약한 hue |
| 평가 / 그림 신뢰도 | mAP 평가 0.001 / 혼동행렬 0.001·0.25 / 이미지 표시 0.25 |
| 후처리 | NMS 없음(YOLO26 기본 one2one 헤드, end2end), max_det=4, 외부 전용 클래스 예측 제거 |
| 성능 | 단일 holdout의 mAP@[0.75:0.95], Precision, Recall, 추론속도와 꺾은선 그래프 |
| 저장 | best.pt, 에폭별 성능·손실 그래프, 이미지별 추론시간 그래프, AP·혼동행렬·GT/예측, 대회 제출 csv |

**검증에서 역전파나 optimizer.step()을 하지 않습니다.**
eval 모드·inference_mode를 적용하고 검증 전후 가중치·버퍼가 같은지도 검사합니다.

실행 시 산출물은 runs/(Colab은 Drive의 MyDrive/pillbbox_runs/) 아래 새 실험 폴더에 모읍니다. 기존 raw·보정 CSV·공용 산출물은
덮어쓰지 않습니다. YOLO 라벨·분할 목록·data.yaml은 data_process.data_main.run()이 만듭니다.

## 0. Colab 준비 (Colab에서만 실행)

Colab이면 Drive를 마운트하고, `/content/pill-bbox`에 이 브랜치를 받고, `uv.lock` 버전으로 패키지를 설치하고,
대회 데이터가 없으면 받습니다. 이미 받은 저장소·데이터는 그대로 씁니다. Mac(로컬)에서는 아무것도 하지 않습니다.
패키지 설치 뒤 커널을 재시작하라는 에러가 나오면 재시작하고 이 셀부터 다시 실행합니다.

In [ ]:
import os
import re
import shutil
import subprocess
import sys
import urllib.request
from getpass import getpass
from importlib import invalidate_caches
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

IN_COLAB = bool(os.getenv("COLAB_RELEASE_TAG"))
COLAB_REPO_URL = "https://github.com/def-uk/pill-bbox.git"
COLAB_BRANCH = "exp/yolo26m_nms_free_aihub"  # origin에 push된 브랜치만 받을 수 있다
COLAB_REPO_DIR = Path("/content/pill-bbox")
# 실험 폴더(best.pt·last.pt·그림·제출 csv)는 Drive에 둬서 런타임이 끊겨도 남긴다.
COLAB_OUTPUT_ROOT = Path("/content/drive/MyDrive/pillbbox_runs")
# 학습 입력(prepared)은 런타임 디스크에 둔다. Drive에는 이미지 링크를 만들 수 없어
# 복사되고 읽기도 느리다. data_main이 저장소 기준 경로로 출력하므로 저장소 안(runs/는
# 커밋 안 됨)에 둔다.
COLAB_PREPARED_ROOT = COLAB_REPO_DIR / "runs" / "prepared"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    if not COLAB_REPO_DIR.exists():
        print(f"저장소를 받습니다: {COLAB_REPO_DIR}, branch={COLAB_BRANCH}")
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                COLAB_BRANCH,
                COLAB_REPO_URL,
                str(COLAB_REPO_DIR),
            ],
            check=True,
        )
    os.chdir(COLAB_REPO_DIR)
    if str(COLAB_REPO_DIR) not in sys.path:
        sys.path.insert(0, str(COLAB_REPO_DIR))
    commit = subprocess.run(
        ["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True, check=True
    )
    print(f"저장소: {COLAB_REPO_DIR} ({commit.stdout.strip()})")
    if sys.version_info < (3, 12):  # noqa: UP036 - Colab 커널의 실제 버전을 검사한다.
        raise RuntimeError(
            "이 프로젝트는 Python 3.12 이상이 필요합니다. Colab 런타임을 확인하세요."
        )

    # Colab 커널 자체에 uv.lock 버전의 패키지를 설치한다(yolo11m AI Hub 노트북과 같은 방법).
    uv = shutil.which("uv")
    if uv is None:
        uv_dir = Path("/tmp/colab-uv")
        uv = str(uv_dir / "uv")
        if not Path(uv).is_file():
            with urllib.request.urlopen("https://astral.sh/uv/install.sh", timeout=60) as response:
                installer = response.read().decode("utf-8")
            subprocess.run(
                ["sh"],
                input=installer,
                text=True,
                check=True,
                env={**os.environ, "UV_UNMANAGED_INSTALL": str(uv_dir)},
            )
    requirements = Path("/tmp/colab-requirements.txt")
    subprocess.run(
        [uv, "export", "--frozen", "--no-dev", "--no-hashes", "--no-emit-project"]
        + ["-o", str(requirements)],
        check=True,
    )
    # uv.lock은 Linux에서 torch·torchvision을 CUDA 13.0 빌드(+cu130)로 고정한다. 이 빌드는
    # GPU 드라이버가 CUDA 13.0 이상을 지원해야 돌므로, 드라이버가 낮으면 Colab에 깔린 torch를 쓴다.
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    found = re.search(r"CUDA Version:\s*(\d+)\.(\d+)", smi.stdout)
    driver_cuda = (int(found[1]), int(found[2])) if found else (0, 0)
    print(f"GPU 드라이버가 지원하는 CUDA: {driver_cuda[0]}.{driver_cuda[1]}")
    if driver_cuda < (13, 0):
        lines = requirements.read_text(encoding="utf-8").splitlines()
        keep = [line for line in lines if not line.startswith(("torch==", "torchvision=="))]
        requirements.write_text("\n".join(keep) + "\n", encoding="utf-8")
        print(
            "드라이버가 CUDA 13.0 미만이라 torch·torchvision은 Colab에 깔린 버전을 그대로 씁니다."
        )
    install = subprocess.run(
        [uv, "pip", "install", "--python", sys.executable, "-r", str(requirements)]
        + ["--extra-index-url", "https://download.pytorch.org/whl/cu130"]
        + ["--index-strategy", "unsafe-best-match", "--break-system-packages"],
        capture_output=True,
        text=True,
        check=False,
    )
    if install.returncode:
        # Colab은 하위 프로세스 출력을 셀에 보여 주지 않으므로 에러를 직접 출력한다.
        print(install.stderr[-3000:])
        raise RuntimeError(
            f"패키지 설치 실패(종료 코드 {install.returncode}). 위 메시지를 확인하세요."
        )
    print(install.stderr.strip().splitlines()[-1] if install.stderr.strip() else "패키지 설치 완료")
    invalidate_caches()
    # Colab이 미리 불러온 패키지는 설치만으로 메모리의 버전이 바뀌지 않는다.
    distributions = {
        "numpy": "numpy",
        "pandas": "pandas",
        "torch": "torch",
        "ultralytics": "ultralytics",
        "matplotlib": "matplotlib",
        "PIL": "pillow",
    }
    stale = []
    for module_name, distribution in distributions.items():
        module = sys.modules.get(module_name)
        if module is None or not hasattr(module, "__version__"):
            continue
        try:
            if str(module.__version__) != version(distribution):
                stale.append(module_name)
        except PackageNotFoundError:
            continue
    if stale:
        raise RuntimeError(
            f"패키지 설치가 끝났지만 메모리에 이전 버전이 남아 있습니다: {stale}. "
            "커널을 재시작한 뒤 이 셀부터 다시 실행하세요."
        )

    # 대회 원본이 없으면 받는다. 링크는 저장소에 남기지 않으려고 입력창으로 받는다.
    raw_root = COLAB_REPO_DIR / "data" / "raw"
    if not any(
        (path / "train_images").is_dir()
        for path in (raw_root, raw_root / "sprint_ai_project1_data")
    ):
        from data_process.download_data import download_data

        download_data(getpass("대회 데이터 Drive 링크 DATA_URL (화면에 표시되지 않음): ").strip())
print(f"Colab={IN_COLAB}")

In [ ]:
import os
import sys
from pathlib import Path

# 루트 또는 notebooks에서 실행해도 프로젝트 import 경로를 찾는다.
PROJECT_PATH = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "pyproject.toml").is_file() and (path / "src" / "config.py").is_file()
    ),
    None,
)
if PROJECT_PATH is None:
    raise RuntimeError("pill-bbox 프로젝트 안에서 노트북을 실행하세요.")
if str(PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_PATH))

os.environ["WANDB_MODE"] = "disabled"
os.environ["YOLO_AUTOINSTALL"] = "false"
# MPS에 없는 연산은 CPU로 넘겨 실행한다. torch를 import하기 전에 정해야 한다.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

In [ ]:
import gc
import json
import platform
import shutil
from copy import copy
from dataclasses import asdict
from datetime import datetime
from getpass import getpass
from time import perf_counter
from unittest.mock import patch
from zoneinfo import ZoneInfo

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from dotenv import load_dotenv
from IPython import get_ipython
from IPython.display import Image, display
from matplotlib.patches import Rectangle
from sklearn.model_selection import GroupShuffleSplit
from ultralytics import YOLO
from ultralytics.cfg import DEFAULT_CFG_DICT
from ultralytics.models.yolo.detect import DetectionValidator
from ultralytics.utils.metrics import ConfusionMatrix
from ultralytics.utils.torch_utils import EarlyStopping, init_seeds

from data_process import data_yaml
from data_process.aihub_data import (
    download_drive_folder,
    extract_zips,
    find_combo_zips,
    list_combo_images,
    load_combo_annotations,
    zip_key,
)
from data_process.data_processing import (
    bbox_flags,
    find_box_pairs,
    list_images,
    read_image_headers,
)
from data_process.make_labels import (
    LABEL_VERSIONS,
    file_sha256,
    read_yolo_labels,
    write_yolo_labels,
)
from data_process.move_files import link_images
from data_process.split import FOLD_COLS, RARE_MAX, SplitSpec, rare_classes
from models.evaluate import CONFUSION_PAIRS, MAIN_METRIC, ap75_95, map75_95, size_group
from models.train import AUG_PRESETS, MAIN_METRIC_KEY, Map7595Trainer, is_finished
from src.config import PROJECT_ROOT, RUNS_DIR, get_device

## 1. 설정과 장치

full fine tuning을 사용합니다. 그 밖의 미지정 학습 인자는 Ultralytics 기본값이며 실제 인자를
train/args.yaml에 남깁니다. lr0는 시작 학습률입니다. warmup·스케줄과 MuSGD의 일부
head 그룹 3배 배율은 유지하고 실제 그룹별 학습률을 기록합니다.

`NMS_ENABLED=True`면 one2many 헤드 + 클래스별 NMS(IoU=0.5), `False`면 YOLO26 기본인
one2one 헤드(NMS 없음, end2end)로 학습 중 검증·최종 평가·속도 측정을 모두 합니다.
`AIHUB_*`는 외부 데이터 설정입니다. 외부 전용 알약을 새 클래스로 학습하므로 모델 클래스 수는
대회 56종보다 많습니다. 학습 중 검증·최종 평가·제출에서는 외부 전용 클래스 예측을 지우고
대회 56종만 남깁니다(max_det=4 안에서 지우므로 이미지당 박스가 4개보다 적을 수 있습니다).

최종 평가·시각화의 기반 예측·속도 측정은 FP32를 쓰며, 학습 AMP는 기본 True이며 MPS에서는 ultralytics가 자동으로 끕니다.

In [ ]:
MODEL_NAME = "yolo26m.pt"
LABEL_VERSION = "v1"
SEED = 42
VAL_FRACTION = 0.2  # 희귀 조합을 제외한 나머지 조합의 validation 비율.
PIPELINE_PARTITION_ID = 0  # 공용 파일 형식에 필요한 식별자이며 교차검증 번호가 아니다.
SPLIT_NAME = f"combo_{LABEL_VERSION}_s{SEED}"
EPOCHS = 100
PATIENCE = 10
LR0 = 0.001
OPTIMIZER = "MuSGD"
BATCH = 16
IMGSZ = 640
FREEZE = None
AUG = "on"
EVAL_CONF = 0.001
CM_CONFS = (0.001, 0.25)
VIS_CONF = 0.25
CM_MATCH_IOU = 0.45  # 라이브러리 기본 정답 매칭 기준. NMS IoU와 구분한다.
NMS_IOU = 0.5
NMS_ENABLED = False
MAX_DET = 4
GPU_INDEX = 0  # CUDA에서만 사용한다.
# 이미지 읽기·증강을 하는 프로세스 수. 0이면 학습과 같은 프로세스에서 차례로 처리해
# AI Hub를 더한 학습(사진 29배)에서 병목이 된다. Windows에서 멈추면 0으로 되돌린다.
WORKERS = 8
VERIFY_VALIDATION_WEIGHTS = True
BENCHMARK_WARMUP = 5
BENCHMARK_ROUNDS = 3
# 장치는 get_device()로 고른다(CUDA > MPS). CUDA·MPS 모두 본 학습을 할 수 있다.
DEVICE_TYPE = get_device()
SMOKE = False  # True: 동작 확인(1 epoch, 학습 데이터 10%). 본 학습은 False
SMOKE_EPOCHS = 1
SMOKE_FRACTION = 0.1
TRAIN_EPOCHS = SMOKE_EPOCHS if SMOKE else EPOCHS
TRAIN_FRACTION = SMOKE_FRACTION if SMOKE else 1.0

# AI Hub 외부 데이터 (02_eda §5~8). train에만 더하고 validation은 대회 사진 그대로 둔다.
USE_AIHUB = True  # False: 원본 NMS-free 노트북과 같은 대회 데이터만 학습
# 02_eda §5: 클래스당 bbox 목표. 대회 최다 클래스 수(157)라 공통 클래스 전부를 여기까지 채운다.
AIHUB_TARGET_BBOX = 157
# 쓰지 않는 zip·폴더 구분자 (data/README.md). yolo11m AI Hub 노트북과 같다.
AIHUB_EXCLUDE = ["TL_2", "TS_2"]
# 외부 데이터 폴더가 없을 때 조합 zip을 받을 팀 Drive 공유 폴더 링크(aihub_pill).
# 비워 두면 받아야 할 때 입력창으로 묻는다. AI Hub 약관상 팀원에게만 공유하는 링크라
# 넣은 채로 커밋하지 않는다.
AIHUB_URL = ""
# zip을 이미 받아 둔 폴더. 예: Colab에서 공유 폴더를 내 Drive에 바로가기로 추가한 경로
# Path("/content/drive/MyDrive/aihub_pill"). None이면 AIHUB_URL에서
# data/external/aihub_zips로 받는다.
AIHUB_ZIP_DIR = None
AIHUB_TAG = f"_aihub{AIHUB_TARGET_BBOX}cls" if USE_AIHUB else ""

# 결과 폴더·제출 파일 이름에 넣을 설정 태그.
# 예: yolo26m_nmsfree_MuSGD_lr0.001_e100_p10_b16_img640_augon_det4_aihub157cls
RUN_TAG = (
    f"{Path(MODEL_NAME).stem}_{'nms' if NMS_ENABLED else 'nmsfree'}_{OPTIMIZER}_lr{LR0:g}"
    f"_e{TRAIN_EPOCHS}_p{PATIENCE}_b{BATCH}_img{IMGSZ}_aug{AUG}_det{MAX_DET}{AIHUB_TAG}"
)
# 실험 폴더를 모으는 곳. Colab은 Drive(0절의 COLAB_OUTPUT_ROOT), 로컬은 프로젝트 runs/.
OUTPUT_ROOT = COLAB_OUTPUT_ROOT if IN_COLAB else RUNS_DIR
# 기존 실험 폴더(이름만 적으면 OUTPUT_ROOT 아래에서 찾음). 학습이 끝난 실험이면 학습을 건너뛰고
# best.pt로 평가·제출 csv만 만들고, 중간에 끊긴 실험이면 train/weights/last.pt에서 이어 학습한다.
# 준비 파일(prepared)이 없으면(Colab 새 런타임) 같은 설정으로 다시 만들고 이전과 같은지 검사한다.
# 예: REUSE_DIR = "yolo26m_nmsfree_..._combo_holdout_s42_20261102_103918_980117"
REUSE_DIR = None
run_stamp = datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y%m%d_%H%M%S_%f")
if REUSE_DIR:
    EXPERIMENT_DIR = Path(REUSE_DIR).expanduser()
    if not EXPERIMENT_DIR.is_absolute():
        EXPERIMENT_DIR = OUTPUT_ROOT / EXPERIMENT_DIR
else:
    EXPERIMENT_DIR = (
        OUTPUT_ROOT / f"{'smoke_' if SMOKE else ''}{RUN_TAG}_combo_holdout_s{SEED}_{run_stamp}"
    )
PREPARED_DIR = (
    COLAB_PREPARED_ROOT / EXPERIMENT_DIR.name if IN_COLAB else EXPERIMENT_DIR / "prepared"
)
# Colab은 prepared가 실험 폴더 밖이라 데이터 준비가 실험 폴더를 만들지 않으므로 여기서 만든다.
if not REUSE_DIR:
    EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
PRETRAINED_PATH = EXPERIMENT_DIR / "pretrained" / MODEL_NAME

EXPERIMENT_CONFIG = {
    "model": MODEL_NAME,
    "label": LABEL_VERSION,
    "split": SPLIT_NAME,
    "split_strategy": "single_holdout",
    "cross_validation": False,
    "n_splits": 1,
    "validation_fraction_free_combos": VAL_FRACTION,
    "seed": SEED,
    "run_tag": RUN_TAG,
    "epochs": TRAIN_EPOCHS,
    "fraction": TRAIN_FRACTION,
    "smoke": SMOKE,
    "device_type": DEVICE_TYPE,
    "patience": PATIENCE,
    "optimizer": OPTIMIZER,
    "lr0": LR0,
    "batch": BATCH,
    "imgsz": IMGSZ,
    "transfer": "full fine tuning",
    "freeze": FREEZE,
    "augmentation": AUG_PRESETS[AUG],
    "main_metric": MAIN_METRIC,
    "eval_conf": EVAL_CONF,
    "cm_confs": list(CM_CONFS),
    "vis_conf": VIS_CONF,
    "cm_match_iou": CM_MATCH_IOU,
    "nms": NMS_ENABLED,
    "nms_iou": NMS_IOU,
    "agnostic_nms": False,
    "max_det": MAX_DET,
    "rare_max": RARE_MAX,
    "split_unit": "combo",
    "fix_rare_train": True,
    "workers": WORKERS,
    "verify_validation_weights": VERIFY_VALIDATION_WEIGHTS,
    "evaluation_precision": "FP32",
    "benchmark_batch": 1,
    "benchmark_warmup": BENCHMARK_WARMUP,
    "benchmark_rounds": BENCHMARK_ROUNDS,
    "aihub": USE_AIHUB,
    "aihub_train_only": True,
    "aihub_target_bbox": AIHUB_TARGET_BBOX,
    "aihub_exclude": list(AIHUB_EXCLUDE),
    "aihub_selection": "02_eda §7 (선별 후 검사)",
    "aihub_whole_image": True,
    "aihub_extra_pills": "new_class",
    "aihub_problem_images": "exclude",
    "aihub_opencv_review": "not_applied",
    "aihub_competition_combo": "exclude",
}
# 학습 결과에 영향이 없는 기록·평가용 항목. 재사용할 때 이 항목은 달라도 된다.
# conf·NMS·max_det는 학습 중 검증(best.pt 선택)에 쓰였으므로 같아야 한다.
NON_TRAINING_KEYS = {
    "run_tag",
    "smoke",
    "device_type",
    "workers",
    "verify_validation_weights",
    "evaluation_precision",
    "benchmark_batch",
    "benchmark_warmup",
    "benchmark_rounds",
    "cm_confs",
    "vis_conf",
    "cm_match_iou",
}


def training_config_diff(saved: dict) -> list[str]:
    """저장된 실험 설정과 현재 설정에서 학습에 영향을 주는 항목 중 값이 다른 것을 반환한다."""
    current = json.loads(json.dumps(EXPERIMENT_CONFIG))
    return sorted(
        key
        for key in saved.keys() & current.keys()
        if key not in NON_TRAINING_KEYS and saved[key] != current[key]
    )


if REUSE_DIR:
    saved_path = EXPERIMENT_DIR / "experiment.json"
    if not saved_path.is_file():
        raise FileNotFoundError(f"재사용할 실험 기록이 없습니다: {saved_path}")
    saved_config = json.loads(saved_path.read_text(encoding="utf-8"))["config"]
    if diff := training_config_diff(saved_config):
        raise ValueError(
            "재사용할 실험과 학습 설정이 다릅니다. 설정 셀을 아래 값에 맞추세요: "
            + ", ".join(f"{key}={saved_config[key]!r}" for key in diff)
        )
    print(f"기존 실험 재사용: {EXPERIMENT_DIR.name} (끝난 학습은 평가만, 끊긴 학습은 이어서)")
display(pd.Series(EXPERIMENT_CONFIG, name="실험 설정").to_frame())

In [ ]:
# CUDA·MPS가 모두 없으면 CPU로 자동 대체하지 않고 중단한다.
if get_device() != DEVICE_TYPE:
    raise RuntimeError("설정 셀 이후 장치가 바뀌었습니다. 설정 셀부터 다시 실행하세요.")
if DEVICE_TYPE not in {"cuda", "mps"}:
    raise RuntimeError(
        "CUDA GPU나 Apple MPS를 사용할 수 없습니다. 프로젝트 uv 커널과 "
        "CUDA 지원 PyTorch(NVIDIA) 또는 MPS 지원 PyTorch(Mac)를 확인하세요."
    )
if DEVICE_TYPE == "cuda" and not 0 <= GPU_INDEX < torch.cuda.device_count():
    raise ValueError(f"GPU_INDEX={GPU_INDEX}, GPU 수={torch.cuda.device_count()}")
if not {"nms", "quantize"} <= DEFAULT_CFG_DICT.keys():
    raise RuntimeError("프로젝트의 Ultralytics 8.4.182 이상 환경을 선택하세요.")
if LABEL_VERSION != "v1" or VIS_CONF not in CM_CONFS:
    raise ValueError("이 베이스라인은 v1과 혼동행렬에 포함된 표시 신뢰도를 사용합니다.")
if AIHUB_TARGET_BBOX < 1:
    raise ValueError("AIHUB_TARGET_BBOX는 1 이상이어야 합니다.")
if not 0 < VAL_FRACTION < 1 or PIPELINE_PARTITION_ID != 0:
    raise ValueError("단일 holdout 비율과 공용 파이프라인 식별자를 확인하세요.")
if DEVICE_TYPE == "cuda":
    torch.cuda.set_device(GPU_INDEX)
    DEVICE = f"cuda:{GPU_INDEX}"
    gpu = torch.cuda.get_device_properties(GPU_INDEX)
    gpu_name, gpu_memory = gpu.name, gpu.total_memory
else:
    DEVICE = "mps"  # ultralytics는 mps:0이 아니라 mps를 받는다.
    gpu_name = f"Apple MPS ({platform.machine()})"
    gpu_memory = torch.mps.recommended_max_memory()
init_seeds(SEED, deterministic=True)


def sync_device() -> None:
    """속도 측정 전후로 장치의 비동기 연산이 끝날 때까지 기다린다."""
    if DEVICE_TYPE == "cuda":
        torch.cuda.synchronize(DEVICE)
    else:
        torch.mps.synchronize()


def empty_device_cache() -> None:
    """모델을 지운 뒤 장치 메모리 캐시를 비운다."""
    if DEVICE_TYPE == "cuda":
        torch.cuda.empty_cache()
    else:
        torch.mps.empty_cache()


ENVIRONMENT = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "cuda": torch.version.cuda,
    "device": DEVICE,
    "gpu": gpu_name,
    "gpu_memory_gib": round(gpu_memory / 1024**3, 2),
    "smoke": SMOKE,
}
raw_root = PROJECT_ROOT / "data" / "raw"
RAW_DIR = raw_root / "sprint_ai_project1_data"
if not RAW_DIR.is_dir():
    RAW_DIR = raw_root
TRAIN_IMAGES = list_images(RAW_DIR / "train_images")
TEST_IMAGES = list_images(RAW_DIR / "test_images")
if (len(TRAIN_IMAGES), len(TEST_IMAGES)) != (232, 842):
    raise ValueError(
        f"확인한 원본 수와 다릅니다: train={len(TRAIN_IMAGES)}, test={len(TEST_IMAGES)}"
    )
if len({path.name for path in TRAIN_IMAGES}) != len(TRAIN_IMAGES):
    raise ValueError("중복된 train 이미지 파일명이 있습니다.")
display(pd.Series(ENVIRONMENT, name="장치 환경").to_frame())
print(f"결과 폴더: {EXPERIMENT_DIR}")

## 2. EDA 보정 라벨과 희귀 조합 train 고정

현재 v1 파이프라인은 누락 8개 추가, kwon 좌표 수정, 검수된 19232 캡슐 보정을 사용합니다.
EDA의 과거 미결정 설명과 현재 파이프라인 적용 상태는 다릅니다. 현재 v1을 따르며
여백 보정(v2)이나 새 검수 승인은 하지 않습니다.

분할 단위는 파일 이름 첫 밑줄 앞의 **알약 조합(combo)**입니다. 같은 조합의 모든 각도 사진은
train 또는 validation 한쪽에만 둡니다. 촬영 세팅이 비슷해도 조합이 다르면 따로 분할합니다.

보정 후 전체 bbox가 5개 이하인 클래스가 든 조합 전체를 train에 고정합니다.
나머지 조합은 GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)로
**한 번만 분할**합니다. 분할 후보나 교차검증 fold를 생성하지 않습니다.
8:2는 고정분을 제외한 조합 수 기준이며 조합별 이미지 수가 달라 실제 이미지 비율은 다릅니다.

라벨·YOLO 변환·목록 파일 생성은 기존 data_main.run()을 사용하고, 그 호출 동안만
분할 함수를 아래 단일 holdout 함수로 연결합니다. 실제 방식은 meta.json과 experiment.json에
기록합니다. 공용 파이프라인 파일 형식상 fold_0.csv와 fold_0/data.yaml이라는 이름을 쓰지만,
이는 하나의 분할을 저장하는 식별자일 뿐 K-fold 교차검증을 의미하지 않습니다.

In [ ]:
def write_json(path: Path, content: dict | list) -> None:
    """실험 기록을 UTF-8 JSON으로 저장한다."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(content, ensure_ascii=False, indent=2), encoding="utf-8")


def audit_holdout(labels: pd.DataFrame, table: pd.DataFrame) -> dict:
    """단일 분할의 이미지 보존·조합 누수·희귀 조합 train 고정을 검사한다."""
    required = set(FOLD_COLS)
    if table.empty or not required <= set(table.columns):
        raise AssertionError("단일 holdout 분할 표가 없거나 필수 열이 빠졌습니다.")
    if set(table["fold"]) != {PIPELINE_PARTITION_ID}:
        raise AssertionError("공용 파이프라인에는 단일 분할 표 하나만 전달해야 합니다.")
    if table["file_name"].duplicated().any():
        raise AssertionError("분할 표에 중복 이미지가 있습니다.")
    if set(table["file_name"]) != set(labels["file_name"]):
        raise AssertionError("분할 표와 라벨의 이미지가 다릅니다.")
    if set(table["split"]) != {"train", "val"}:
        raise AssertionError("train/validation이 모두 필요합니다.")
    expected_combo = table["file_name"].str.split("_").str[0]
    if not table["combo"].eq(expected_combo).all() or not table["group"].eq(table["combo"]).all():
        raise AssertionError("이미지를 combo 단위로 묶어야 합니다.")
    if table.groupby("combo")["split"].nunique().gt(1).any():
        raise AssertionError("같은 combo가 train/validation 양쪽에 있습니다.")
    rare_ids = set(rare_classes(labels))
    rare_files = labels.loc[labels["category_id"].isin(rare_ids), "file_name"]
    rare_combos = set(rare_files.str.split("_").str[0])
    should_fix = table["combo"].isin(rare_combos)
    if not table["fixed"].eq(should_fix).all():
        raise AssertionError("희귀 알약이 포함된 조합 전체를 고정해야 합니다.")
    if not table.loc[should_fix, "split"].eq("train").all():
        raise AssertionError("희귀 조합이 validation에 포함되었습니다.")
    train_files = set(table.loc[table["split"] == "train", "file_name"])
    val_files = set(table.loc[table["split"] == "val", "file_name"])
    train_ids = set(labels.loc[labels["file_name"].isin(train_files), "category_id"])
    val_ids = set(labels.loc[labels["file_name"].isin(val_files), "category_id"])
    if train_files & val_files or rare_ids & val_ids:
        raise AssertionError("이미지가 중복되거나 희귀 클래스가 validation에 있습니다.")
    free = table.loc[~should_fix]
    return {
        "train_images": len(train_files),
        "val_images": len(val_files),
        "train_combos": int(table.loc[table["split"] == "train", "combo"].nunique()),
        "val_combos": int(table.loc[table["split"] == "val", "combo"].nunique()),
        "free_combos": int(free["combo"].nunique()),
        "fixed_train_images": int(should_fix.sum()),
        "fixed_train_groups": len(rare_combos),
        "val_image_ratio": len(val_files) / len(table),
        "free_val_combo_ratio": free.loc[free["split"] == "val", "combo"].nunique()
        / free["combo"].nunique(),
        "val_only_classes": sorted(int(value) for value in val_ids - train_ids),
        "n_classes_not_in_val": len(set(labels["category_id"]) - val_ids),
        "combo_leak": 0.0,
        "rare_in_val": 0,
    }


def build_combo_holdout(
    labels: pd.DataFrame,
    spec: SplitSpec,
    labels_sha256: str,
    val_fraction: float = VAL_FRACTION,
) -> tuple[pd.DataFrame, dict]:
    """희귀 조합을 train에 고정하고 남은 조합을 한 번만 약 8:2로 나눈다."""
    if spec.unit != "combo" or not 0 < val_fraction < 1:
        raise ValueError("combo 단위와 0~1 사이 validation 비율이 필요합니다.")
    units = pd.DataFrame({"file_name": sorted(labels["file_name"].unique())})
    units["combo"] = units["file_name"].str.split("_").str[0]
    units["group"] = units["combo"]
    rare_ids = rare_classes(labels)
    rare_files = labels.loc[labels["category_id"].isin(rare_ids), "file_name"]
    rare_combos = set(rare_files.str.split("_").str[0])
    units["fixed"] = units["combo"].isin(rare_combos)
    free = units.loc[~units["fixed"]].reset_index(drop=True)
    if free["combo"].nunique() < 2:
        raise ValueError("희귀 조합을 제외한 조합이 2개 이상이어야 단일 분할할 수 있습니다.")
    splitter = GroupShuffleSplit(n_splits=1, test_size=val_fraction, random_state=spec.seed)
    _, val_indices = next(splitter.split(free, groups=free["combo"]))
    val_files = set(free.iloc[val_indices]["file_name"])
    table = units.assign(
        fold=PIPELINE_PARTITION_ID,
        split=units["file_name"].isin(val_files).map({True: "val", False: "train"}),
    )[FOLD_COLS]
    check = audit_holdout(labels, table)
    meta = {
        **asdict(spec),
        "fix_rare": True,
        "strategy": "single_holdout",
        "cross_validation": False,
        "n_splits": 1,
        "validation_fraction_free_combos": val_fraction,
        "rare_max": RARE_MAX,
        "rare_classes": rare_ids,
        "labels_sha256": labels_sha256,
        "n_images": len(units),
        "fixed_train_images": check["fixed_train_images"],
        "fixed_train_groups": check["fixed_train_groups"],
        "created_at": datetime.now().astimezone().isoformat(timespec="seconds"),
        # step3_split()의 기록 형식에만 필요한 단일 파티션 식별자.
        "checks": [{"fold": PIPELINE_PARTITION_ID, **check}],
    }
    return table, meta


def prepare_data() -> Path:
    """기존 파이프라인으로 단일 holdout의 학습 입력 하나만 준비한다."""
    import matplotlib

    previous_backend = matplotlib.get_backend()
    try:
        from data_process import data_main
    finally:
        plt.switch_backend(previous_backend)
        # data_main은 import할 때 Agg로 바꾸고, 이때 IPython의 그림 PNG 변환기가 해제된다.
        # 백엔드만 되돌리면 plt.show()가 <Figure ...> 글자만 내므로 inline 표시를 다시 켠다.
        shell = get_ipython()
        if shell is not None and "inline" in previous_backend:
            shell.run_line_magic("matplotlib", "inline")
    if PREPARED_DIR.exists():
        raise FileExistsError(
            "이미 준비한 실험입니다. 설정 셀에서 새 실험을 만들거나 다음 셀로 진행하세요."
        )
    source_dir = PROJECT_ROOT / "data" / "processed"
    required = ("added_annotations.csv", "fixed_annotations.csv")
    for name in required:
        if not (source_dir / name).is_file():
            raise FileNotFoundError(f"기존 보정 기록이 없습니다: {source_dir / name}")
    PREPARED_DIR.mkdir(parents=True)
    for name in (*required, "padding_review.csv"):
        if (source_dir / name).is_file():
            shutil.copy2(source_dir / name, PREPARED_DIR / name)
    # 기존 소스는 수정하지 않고, run()의 출력 경로와 단일 분할 함수를 호출 동안 연결한다.
    with (
        patch.object(data_main, "PROCESSED_DIR", PREPARED_DIR),
        patch.object(data_yaml, "YOLO_ROOT", PREPARED_DIR / "yolo"),
        patch.object(data_main, "build_split", build_combo_holdout),
    ):
        paths = data_main.run(
            LABEL_VERSION, [SPLIT_NAME], use_code_names=True, n_check=0, seed=SEED
        )
        expected = data_yaml.data_yaml_path(LABEL_VERSION, SPLIT_NAME, PIPELINE_PARTITION_ID)
        if paths != [expected]:
            raise AssertionError(f"단일 holdout data.yaml 하나가 필요합니다: {paths}")
    return expected

In [ ]:
# 실행 시 데이터 준비 파일만 생성한다. 학습은 4절의 학습 셀에서 시작한다.
# REUSE_DIR이면 그 실험에서 만든 준비 파일을 그대로 쓴다. 없으면(Colab 새 런타임) 다시 만든다.
if REUSE_DIR:
    with patch.object(data_yaml, "YOLO_ROOT", PREPARED_DIR / "yolo"):
        DATA_YAML = data_yaml.data_yaml_path(LABEL_VERSION, SPLIT_NAME, PIPELINE_PARTITION_ID)
    if not DATA_YAML.is_file():
        print(f"준비 파일이 없어 같은 설정으로 다시 만듭니다: {PREPARED_DIR}")
        DATA_YAML = prepare_data()
else:
    DATA_YAML = prepare_data()
LABELS = pd.read_csv(PREPARED_DIR / "labels" / f"labels_{LABEL_VERSION}.csv")
CLASS_MAP = pd.read_csv(PREPARED_DIR / "yolo" / LABEL_VERSION / "class_map.csv")
SPLIT_TABLE = pd.read_csv(
    PREPARED_DIR / "splits" / SPLIT_NAME / f"fold_{PIPELINE_PARTITION_ID}.csv"
)
SPLIT_META = json.loads(
    (PREPARED_DIR / "splits" / SPLIT_NAME / "meta.json").read_text(encoding="utf-8")
)
if set(LABELS["file_name"]) != {path.name for path in TRAIN_IMAGES}:
    raise AssertionError("v1 라벨과 원본 train 이미지가 다릅니다.")
SPLIT_AUDIT = audit_holdout(LABELS, SPLIT_TABLE)
if REUSE_DIR:
    # 다시 만든 라벨·분할이 처음 실험과 같아야 이어 학습·평가 결과를 믿을 수 있다.
    saved_split = json.loads((EXPERIMENT_DIR / "experiment.json").read_text(encoding="utf-8"))[
        "split"
    ]
    for key in ("labels_sha256", "checks"):
        if json.loads(json.dumps(SPLIT_META[key])) != saved_split[key]:
            raise AssertionError(f"재사용 실험과 라벨·분할이 다릅니다({key}). 데이터를 확인하세요.")
display(pd.Series(SPLIT_AUDIT, name="단일 holdout 검사").to_frame())
print("교차검증 없음: combo 단위의 train/validation 단일 분할")
print(f"적용 라벨: {LABEL_VERSION}, bbox={len(LABELS)}, 클래스={len(CLASS_MAP)}")
print(f"희귀 클래스(bbox {RARE_MAX}개 이하): {rare_classes(LABELS)}")
print(
    f"희귀 조합 train 고정: {SPLIT_META['fixed_train_images']}장 / "
    f"{SPLIT_META['fixed_train_groups']}조합"
)
print(
    f"실제 이미지당 최대 정답 수: {LABELS.groupby('file_name').size().max()} / 탐지 제한: {MAX_DET}"
)
print("train 전용 희귀 클래스의 validation AP는 평가 불가(NaN)로 기록합니다.")
pd.DataFrame([SPLIT_AUDIT]).to_csv(EXPERIMENT_DIR / "holdout_audit.csv", index=False)
if not REUSE_DIR:  # 재사용할 때는 원래 실험 기록을 덮어쓰지 않는다.
    write_json(
        EXPERIMENT_DIR / "experiment.json",
        {
            "config": EXPERIMENT_CONFIG,
            "environment": ENVIRONMENT,
            "split": SPLIT_META,
            "label_corrections": list(LABEL_VERSIONS[LABEL_VERSION]),
            "source_correction_sha256": {
                path.name: file_sha256(path) for path in PREPARED_DIR.glob("*.csv")
            },
        },
    )

## 2-2. AI Hub 외부 데이터 추가 (02_eda §5~8)

`02_eda.ipynb`와 같은 방법·seed로 외부 bbox를 고르고, 그 bbox가 든 사진을 **train에만** 더합니다.
validation은 위에서 만든 대회 사진 그대로입니다. 코드는 02_eda §1·2·5~8과 팀원 yolo11m AI Hub
노트북의 선별·검사를 이 노트북 안에 옮긴 것입니다.

1. **선별 (§5~7)**: 대회 v1과 공통인 클래스마다 `AIHUB_TARGET_BBOX`(157)개가 되도록 부족한 수만큼
   seed 42로 bbox를 뽑습니다. 좌표 오류·같은 약 중복 라벨이 있는 사진, 대회 train·test와 같은
   이름의 사진은 후보에서 뺍니다(02_eda §6과 같음).
2. **문제 사진 제외 (§8)**: 파일명의 약 코드와 라벨이 다르거나(라벨 누락·파일명에 없는 라벨),
   bbox끼리 겹치거나(IoU > 0), 좌표가 잘못되었거나, JSON의 사진 크기와 실제 크기가 다른 사진은 뺍니다.
   02_eda §10~11의 OpenCV 보정은 사람 검수 전이라 적용하지 않습니다.
3. **대회와 같은 조합 제외**: 대회 사진과 약 조합이 같은 외부 사진은 같은 조합의 validation 사진과
   겹칠 수 있으므로 뺍니다.
4. 1~3은 팀원 yolo11m AI Hub 노트북(`yolo11m_100_aihub_not_background.ipynb`)과 같습니다.
5. **라벨**: 사진을 통째로 쓰므로 함께 찍힌 알약도 모두 라벨링합니다. 대회 56종에 없는 외부 전용
   알약은 대회 번호(0~55) 뒤에 이어지는 **새 클래스**로 학습합니다.
6. **YOLO 변환**: 외부 사진 링크와 txt는 `prepared/yolo/v1_aihub/`에 따로 만들고 대회 파이프라인
   폴더(`prepared/yolo/v1`)는 건드리지 않습니다. `train.txt` = 대회 train + 외부 사진,
   `val.txt`는 대회 파이프라인 파일 그대로, `nc`·`names`는 외부 전용 클래스만큼 늘립니다.

외부 데이터는 푼 폴더(`.env`의 `AIHUB_EXTRACT_DIR`, 없으면 `data/external/aihub_combo`)에서 읽습니다.
그 폴더가 없거나 비어 있으면 팀 Drive 공유 폴더에서 조합 zip을 `data/external/aihub_zips/`로 받아
`AIHUB_EXCLUDE`를 뺀 zip만 풉니다. 링크는 설정 셀의 `AIHUB_URL`, 비어 있으면 입력창으로 받습니다.
zip을 이미 받아 둔 폴더가 있으면 설정 셀의 `AIHUB_ZIP_DIR`에 넣어 다운로드를 건너뜁니다.
링크는 AI Hub 약관상 팀원에게만 공유하므로(data/README.md) 넣은 채로 커밋하지 않습니다.
쓴 bbox·라벨·제외 사유 표는 `prepared/yolo/v1_aihub/`의 csv와 `aihub_info.json`,
`experiment.json`의 `aihub`에 남습니다.


In [ ]:
AIHUB_BOX_NUMBERS = ["x", "y", "w", "h", "width", "height"]
AIHUB_YOLO_DIR = PREPARED_DIR / "yolo" / f"{LABEL_VERSION}_aihub"
TRAIN_DATA_YAML_PATH = (
    AIHUB_YOLO_DIR / "splits" / SPLIT_NAME / f"fold_{PIPELINE_PARTITION_ID}" / "data.yaml"
)
N_COMP_CLASSES = len(CLASS_MAP)  # yolo_idx가 이 값 이상이면 외부 전용 클래스


def combo_of(file_name: str) -> str:
    """파일명 첫 밑줄 앞의 알약 조합(K-001900-003544-...)."""
    return file_name.split("_")[0]


def combo_codes(file_name: str) -> set[int]:
    """파일명 조합의 약 코드 집합. 사진에서 센 알약 수가 아니다."""
    return {int(code) for code in combo_of(file_name).split("-")[1:]}


def invalid_boxes(boxes: pd.DataFrame) -> pd.Series:
    """좌표 결측·크기 0 이하·이미지 경계 초과·같은 약 중복 라벨인 bbox (02_eda §6)."""
    numeric = boxes[AIHUB_BOX_NUMBERS].apply(pd.to_numeric, errors="coerce")
    flags = bbox_flags(boxes)
    return (
        ~np.isfinite(numeric).all(axis=1)
        | numeric[["width", "height"]].le(0).any(axis=1)
        | flags["out_of_image"]
        | flags["non_positive"]
        | boxes.duplicated(["file_name", "category_id"], keep=False)
    )


def exclusion_reasons(boxes: pd.DataFrame, paths: pd.Series, comp_combos: set[str]) -> pd.DataFrame:
    """외부 사진마다 train에서 뺄 사유를 표로 만든다 (02_eda §8, yolo11m AI Hub 노트북과 같음)."""
    files = sorted(boxes["file_name"].unique())
    boxes = boxes.astype({"category_id": int})
    numeric = boxes[AIHUB_BOX_NUMBERS].apply(pd.to_numeric, errors="coerce")
    positive = np.isfinite(numeric).all(axis=1) & numeric[["w", "h", "width", "height"]].gt(0).all(
        axis=1
    )
    pairs = find_box_pairs(boxes[positive], max_gap=0)
    labeled = boxes.groupby("file_name")["category_id"].agg(set)
    # 헤더 표의 file_name은 디스크 이름(NFD일 수 있음)이라 순서로 맞춘다.
    headers = read_image_headers([Path(paths[name]) for name in files], "aihub")
    real_size = headers[["width", "height"]].to_numpy(dtype=float)
    json_size = boxes.groupby("file_name")[["width", "height"]].first().loc[files]
    reasons = pd.DataFrame(index=pd.Index(files, name="file_name"))
    reasons["라벨 누락"] = [bool(combo_codes(name) - labeled[name]) for name in files]
    reasons["파일명에 없는 라벨"] = [bool(labeled[name] - combo_codes(name)) for name in files]
    reasons["bbox 겹침"] = reasons.index.isin(set(pairs.loc[pairs["iou"].gt(0), "file_name"]))
    reasons["좌표·중복 오류"] = reasons.index.isin(
        set(boxes.loc[invalid_boxes(boxes), "file_name"])
    )
    reasons["JSON·실제 크기 다름"] = ~(real_size == json_size.to_numpy(dtype=float)).all(axis=1)
    # 대회 사진과 약 조합이 같으면 같은 조합의 대회 validation 사진과 겹칠 수 있다
    reasons["대회와 같은 조합"] = [combo_of(name) in comp_combos for name in files]
    return reasons


def sample_supplement(pool: pd.DataFrame, needed: pd.Series) -> pd.DataFrame:
    """클래스마다 부족한 수만큼 SEED로 bbox를 비복원 추출한다 (02_eda §7과 같은 순서)."""
    pool = pool.drop_duplicates(["file_name", "category_id", "x", "y", "w", "h"])
    shuffled = pool.sample(frac=1, random_state=SEED)
    order = shuffled.groupby("category_id").cumcount()
    return (
        shuffled[order.lt(shuffled["category_id"].map(needed).fillna(0))]
        .sort_values(["category_id", "file_name"])
        .reset_index(drop=True)
    )


def env_path(name: str, default: str) -> Path:
    """.env의 경로 값(없으면 default)을 프로젝트 루트 기준 절대 경로로 바꾼다."""
    path = Path(os.getenv(name) or default).expanduser()
    return path if path.is_absolute() else PROJECT_ROOT / path


def extracted_folders(extract_dir: Path) -> list[Path]:
    """extract_dir 아래 다 푼 외부 데이터 폴더 중 AIHUB_EXCLUDE에 없는 것."""
    if not extract_dir.is_dir():
        return []
    return sorted(
        folder
        for folder in extract_dir.iterdir()
        if folder.is_dir()
        and not folder.name.startswith(".")
        and zip_key(folder) not in AIHUB_EXCLUDE
    )


def download_aihub(extract_dir: Path) -> None:
    """팀 Drive 공유 폴더에서 조합 zip을 받아 AIHUB_EXCLUDE를 뺀 것만 푼다.

    설정 셀의 AIHUB_ZIP_DIR이 있으면 받지 않고 그 폴더의 zip을 쓴다. 링크는 AIHUB_URL이고,
    비어 있으면 입력창으로 받는다. 이미 받은 파일·다 푼 zip은 건너뛴다(aihub_data.py).
    """
    if AIHUB_ZIP_DIR:
        zip_dir = Path(AIHUB_ZIP_DIR).expanduser()
    else:
        zip_dir = PROJECT_ROOT / "data" / "external" / "aihub_zips"
        url = AIHUB_URL or getpass("AI Hub zip 폴더 Drive 링크 AIHUB_URL (화면에 표시되지 않음): ")
        if not url.strip():
            raise FileNotFoundError(
                f"외부 데이터 폴더가 없고 받을 주소도 없습니다: {extract_dir}. "
                "팀 Drive aihub_pill 폴더 공유 링크를 넣으세요."
            )
        print(f"외부 데이터가 없어 팀 Drive에서 zip을 받습니다 → {zip_dir}")
        download_drive_folder(url.strip(), zip_dir)
    zips = find_combo_zips(zip_dir, exclude=AIHUB_EXCLUDE)
    if not zips:
        raise FileNotFoundError(
            f"조합 zip이 없습니다: {zip_dir}. Drive 폴더의 파일이 '링크가 있는 모든 사용자'로 "
            "공유되어 있는지 확인하세요."
        )
    extract_zips(zips, extract_dir)


def aihub_folders() -> tuple[Path, list[Path]]:
    """02_eda §1과 같은 위치에서 다 푼 외부 데이터 폴더 중 AIHUB_EXCLUDE에 없는 것을 고른다.

    폴더가 없거나 쓸 하위 폴더가 없으면 팀 Drive에서 받아 푼다(download_aihub).
    """
    load_dotenv(PROJECT_ROOT / ".env")
    extract_dir = env_path("AIHUB_EXTRACT_DIR", "data/external/aihub_combo")
    folders = extracted_folders(extract_dir)
    if not folders:
        download_aihub(extract_dir)
        folders = extracted_folders(extract_dir)
    if not folders:
        raise FileNotFoundError(f"분석할 하위 폴더가 없습니다: {extract_dir}, 제외={AIHUB_EXCLUDE}")
    return extract_dir, folders


def select_aihub() -> dict:
    """02_eda §1·2·5~7로 외부 bbox를 고르고 §8 문제 사진을 빼 train에 더할 라벨을 만든다.

    yolo11m AI Hub 노트북과 같이 OpenCV 보정은 적용하지 않고, 대회와 같은 조합의 사진도 뺀다.
    """
    extract_dir, folders = aihub_folders()
    # §1·2: 이미지와 라벨 연결. 같은 이름이 여러 폴더에 있으면 라벨을 안전하게 붙일 수 없다.
    image_index = list_combo_images(folders)
    if image_index.empty:
        raise FileNotFoundError("외부 이미지가 없습니다. TS 원천 이미지 폴더를 확인하세요.")
    if image_index["file_name"].duplicated().any():
        raise ValueError("외부 이미지 이름이 중복됩니다. 02_eda §1에서 중복 목록을 확인하세요.")
    ann_df, _ = load_combo_annotations(folders)
    if ann_df.empty or ann_df["category_id"].isna().any():
        raise ValueError("외부 라벨이 없거나 약 코드를 해석할 수 없는 bbox가 있습니다 (02_eda §2).")
    available = ann_df[ann_df["file_name"].isin(set(image_index["file_name"]))].copy()
    paths = image_index.set_index("file_name")["path"]

    # §5: 대회 v1과 공통인 클래스마다 목표까지 부족한 bbox 수
    comp_counts = LABELS.groupby("category_id").size()
    comp_ids = {int(value) for value in comp_counts.index}
    common = sorted(comp_ids & set(available["category_id"].astype(int)))
    needed = (AIHUB_TARGET_BBOX - comp_counts).clip(lower=0)

    # §6: 좌표·중복 오류가 있는 사진과 대회 사진과 같은 이름은 후보에서 뺀다
    candidates = available[available["category_id"].isin(common)].merge(
        image_index[["file_name", "path"]], on="file_name", validate="many_to_one"
    )
    invalid_files = set(available.loc[invalid_boxes(available), "file_name"])
    comp_names = {path.name for path in (*TRAIN_IMAGES, *TEST_IMAGES)}
    pool = candidates[~candidates["file_name"].isin(invalid_files | comp_names)]

    # §7: 클래스별 부족량만큼 seed로 비복원 추출
    selected = sample_supplement(pool, needed)

    # 선택 bbox가 든 사진 전체의 라벨. 사진을 통째로 쓰므로 함께 찍힌 bbox도 쓴다.
    boxes = available[available["file_name"].isin(set(selected["file_name"]))].copy()
    boxes["category_id"] = boxes["category_id"].astype(int)

    # §8: 문제 사진과 대회와 같은 조합의 사진은 통째로 뺀다
    comp_combos = {combo_of(name) for name in LABELS["file_name"]}
    comp_combos |= {combo_of(name) for name in comp_names if name.startswith("K-")}
    reasons = exclusion_reasons(boxes, paths, comp_combos)
    kept_files = set(reasons.index[~reasons.any(axis=1)])
    labels = boxes[boxes["file_name"].isin(kept_files)].copy()
    if set(labels.loc[labels["category_id"].isin(comp_ids), "file_name"]) != kept_files:
        raise ValueError("대회 클래스 라벨이 없는 외부 사진이 있습니다.")
    return {
        "extract_dir": extract_dir,
        "folders": folders,
        "n_images": len(image_index),
        "n_labels": len(ann_df),
        "common_classes": common,
        "comp_ids": comp_ids,
        "selected": selected,
        "reasons": reasons,
        "labels": labels,
        "paths": paths,
    }


def read_list(path: Path) -> list[str]:
    """목록 txt의 이미지 경로를 빈 줄 없이 읽는다."""
    return [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def write_aihub_data(selection: dict) -> tuple[Path, dict]:
    """외부 사진 링크·YOLO 라벨을 따로 만들고 train = 대회 train + 외부 사진인 data.yaml을 쓴다."""
    labels = selection["labels"]
    files = sorted(labels["file_name"].unique())
    sources = [Path(selection["paths"][name]) for name in files]
    if any(path.name != name for path, name in zip(sources, files, strict=True)):
        raise ValueError("디스크 파일 이름과 라벨 file_name이 다릅니다(NFD 등). 이름을 확인하세요.")

    # 대회 56종(yolo_idx 0~55) 뒤에 외부 전용 클래스를 약 코드 순서로 잇는다.
    extra_ids = sorted(set(labels["category_id"]) - selection["comp_ids"])
    extra_names = labels.drop_duplicates("category_id").set_index("category_id")["dl_name"]
    extra_map = pd.DataFrame(
        {
            "yolo_idx": range(N_COMP_CLASSES, N_COMP_CLASSES + len(extra_ids)),
            "category_id": extra_ids,
            "dl_name": [extra_names.get(code) for code in extra_ids],
        }
    )
    full_class_map = pd.concat([CLASS_MAP, extra_map], ignore_index=True)
    class_index = dict(
        zip(
            full_class_map["category_id"].astype(int),
            full_class_map["yolo_idx"].astype(int),
            strict=True,
        )
    )
    image_dir = AIHUB_YOLO_DIR / "images" / "train"
    label_dir = AIHUB_YOLO_DIR / "labels" / "train"
    link_counts = link_images(sources, image_dir)
    yolo_counts = write_yolo_labels(labels, files, class_index, label_dir)
    yolo_columns = ["yolo_idx", "x", "y", "w", "h"]
    max_error = 0.0
    for file_name, rows in labels.groupby("file_name"):
        txt_path = label_dir / Path(file_name).with_suffix(".txt").name
        back = read_yolo_labels(txt_path, rows["width"].iloc[0], rows["height"].iloc[0])
        expected = rows.assign(yolo_idx=rows["category_id"].map(class_index))
        if len(back) != len(expected) or expected["yolo_idx"].isna().any():
            raise ValueError(f"외부 YOLO 라벨 개수나 클래스가 다릅니다: {file_name}")
        error = np.abs(
            back[yolo_columns].sort_values(yolo_columns).to_numpy(dtype=float)
            - expected[yolo_columns].sort_values(yolo_columns).to_numpy(dtype=float)
        ).max()
        max_error = max(max_error, float(error))
    if (
        max_error > 0.01
        or yolo_counts["n_dropped"]
        or yolo_counts["n_empty"]
        or yolo_counts["n_lines"] != len(labels)
    ):
        raise ValueError(f"외부 YOLO 라벨 검사 실패: {yolo_counts}, 역변환 오차={max_error}px")

    # train = 대회 train + 외부 사진, val = 대회 파이프라인 val.txt 그대로
    base = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
    comp_train = read_list(Path(base["train"]))
    val_path = Path(base["val"])
    val_names = {Path(line).name for line in read_list(val_path)}
    comp_names = val_names | {Path(line).name for line in comp_train}
    if set(files) & comp_names or {combo_of(name) for name in files} & {
        combo_of(name) for name in val_names
    }:
        raise AssertionError("외부 사진이 대회 사진이나 validation 조합과 겹칩니다.")
    aihub_images = [str(image_dir / name) for name in files]
    out_dir = TRAIN_DATA_YAML_PATH.parent
    out_dir.mkdir(parents=True, exist_ok=True)
    train_txt = out_dir / "train.txt"
    train_txt.write_text(
        "".join(f"{path}\n" for path in comp_train + aihub_images), encoding="utf-8"
    )
    names = {
        **{int(key): value for key, value in base["names"].items()},
        **{int(row.yolo_idx): str(row.category_id) for row in extra_map.itertuples()},
    }
    data = {
        "path": base["path"],
        "train": str(train_txt),
        "val": str(val_path),
        "nc": len(names),
        "names": names,
    }
    header = {
        "label_version": LABEL_VERSION,
        "split_name": SPLIT_NAME,
        "fold": PIPELINE_PARTITION_ID,
        "aihub_images": len(aihub_images),
        "extra_classes": len(extra_ids),
    }
    TRAIN_DATA_YAML_PATH.write_text(
        "".join(f"# {key}: {value}\n" for key, value in header.items())
        + yaml.safe_dump(data, allow_unicode=True, sort_keys=False),
        encoding="utf-8",
    )

    keep_columns = ["file_name", "zip_name", "source_json", "annotation_id", "category_id"]
    keep_columns += ["dl_name", *AIHUB_BOX_NUMBERS]
    selection["selected"][keep_columns].to_csv(
        AIHUB_YOLO_DIR / "aihub_selected.csv", index=False, encoding="utf-8"
    )
    labels[keep_columns].to_csv(AIHUB_YOLO_DIR / "aihub_labels.csv", index=False, encoding="utf-8")
    full_class_map.to_csv(AIHUB_YOLO_DIR / "aihub_class_map.csv", index=False, encoding="utf-8")
    reasons = selection["reasons"]
    reasons.to_csv(AIHUB_YOLO_DIR / "aihub_excluded.csv", encoding="utf-8")
    info = {
        "used": True,
        "source": "notebooks/02_eda.ipynb §1·2·5~8, yolo11m_100_aihub_not_background.ipynb",
        "extract_dir": str(selection["extract_dir"]),
        "folders": [folder.name for folder in selection["folders"]],
        "exclude": list(AIHUB_EXCLUDE),
        "target_bbox": AIHUB_TARGET_BBOX,
        "seed": SEED,
        "external_images": selection["n_images"],
        "external_labels": selection["n_labels"],
        "common_classes": len(selection["common_classes"]),
        "selected_bbox": len(selection["selected"]),
        "selected_images": len(reasons),
        "excluded_images": {key: int(value) for key, value in reasons.sum().items()},
        "excluded_images_total": int(reasons.any(axis=1).sum()),
        "train_images": len(aihub_images),
        "train_labels": len(labels),
        "train_labels_competition": int(labels["category_id"].isin(selection["comp_ids"]).sum()),
        "extra_classes": len(extra_ids),
        "extra_class_ids": [int(code) for code in extra_ids],
        "competition_train_images": len(comp_train),
        "val_images": len(val_names),
        "train_ratio": round((len(comp_train) + len(aihub_images)) / len(comp_train), 1),
        "link_counts": link_counts,
        "yolo_counts": yolo_counts,
        "yolo_max_roundtrip_error_px": max_error,
    }
    write_json(AIHUB_YOLO_DIR / "aihub_info.json", info)
    return TRAIN_DATA_YAML_PATH, info

In [ ]:
# 외부 데이터 준비. USE_AIHUB=False면 대회 데이터만 쓴다.
# REUSE_DIR이면 그 실험에서 만든 외부 라벨·목록을 그대로 쓴다(외부 폴더가 없어도 된다).
# 준비 파일이 없으면(Colab 새 런타임) 같은 방법으로 다시 고르고 처음 실험 기록과 같은지 검사한다.
AIHUB_SAME_KEYS = [
    "selected_bbox",
    "selected_images",
    "excluded_images",
    "train_images",
    "train_labels",
    "extra_class_ids",
    "competition_train_images",
    "val_images",
]
if not USE_AIHUB:
    TRAIN_DATA_YAML = DATA_YAML
    AIHUB_LABELS = pd.DataFrame(columns=["file_name", "category_id"])
    AIHUB_INFO = {"used": False}
    print("USE_AIHUB=False: 외부 데이터 없이 대회 데이터만 학습합니다.")
elif REUSE_DIR and TRAIN_DATA_YAML_PATH.is_file():
    TRAIN_DATA_YAML = TRAIN_DATA_YAML_PATH
    AIHUB_LABELS = pd.read_csv(AIHUB_YOLO_DIR / "aihub_labels.csv")
    AIHUB_INFO = json.loads((AIHUB_YOLO_DIR / "aihub_info.json").read_text(encoding="utf-8"))
elif REUSE_DIR:
    print(f"외부 데이터 준비 파일이 없어 같은 방법으로 다시 만듭니다: {AIHUB_YOLO_DIR}")
    saved_aihub = json.loads((EXPERIMENT_DIR / "experiment.json").read_text(encoding="utf-8"))
    saved_aihub = saved_aihub.get("aihub", {})
    AIHUB_SELECTION = select_aihub()
    TRAIN_DATA_YAML, AIHUB_INFO = write_aihub_data(AIHUB_SELECTION)
    AIHUB_LABELS = AIHUB_SELECTION["labels"]
    if diff := [key for key in AIHUB_SAME_KEYS if AIHUB_INFO.get(key) != saved_aihub.get(key)]:
        raise AssertionError(
            f"재사용 실험과 외부 데이터가 다릅니다: {diff}. 외부 zip을 확인하세요."
        )
else:
    if TRAIN_DATA_YAML_PATH.exists():
        raise FileExistsError(
            "이미 외부 데이터를 준비한 실험입니다. 새 실험은 설정 셀부터 실행하세요."
        )
    AIHUB_SELECTION = select_aihub()
    TRAIN_DATA_YAML, AIHUB_INFO = write_aihub_data(AIHUB_SELECTION)
    AIHUB_LABELS = AIHUB_SELECTION["labels"]
    experiment_path = EXPERIMENT_DIR / "experiment.json"
    experiment = json.loads(experiment_path.read_text(encoding="utf-8"))
    experiment["aihub"] = AIHUB_INFO
    write_json(experiment_path, experiment)

if USE_AIHUB:
    rows = [
        ("외부 이미지 (푼 폴더 전체)", AIHUB_INFO["external_images"]),
        ("외부 bbox 라벨 (푼 폴더 전체)", AIHUB_INFO["external_labels"]),
        ("공통 클래스 (대회 v1 ∩ 외부)", AIHUB_INFO["common_classes"]),
        ("02_eda §7 선택 bbox", AIHUB_INFO["selected_bbox"]),
        ("선택 bbox가 든 사진", AIHUB_INFO["selected_images"]),
        *[(f"  제외: {key}", value) for key, value in AIHUB_INFO["excluded_images"].items()],
        ("제외한 사진 (사유가 여럿이어도 한 번)", AIHUB_INFO["excluded_images_total"]),
        ("train에 더할 외부 사진", AIHUB_INFO["train_images"]),
        ("외부 학습 라벨 bbox (전체)", AIHUB_INFO["train_labels"]),
        ("  그중 대회 클래스", AIHUB_INFO["train_labels_competition"]),
        ("외부 전용 새 클래스 수", AIHUB_INFO["extra_classes"]),
        (
            "대회 train / validation 사진",
            f"{AIHUB_INFO['competition_train_images']} / {AIHUB_INFO['val_images']}",
        ),
        ("train 배수 (에폭당 시간도 대략 이만큼)", AIHUB_INFO["train_ratio"]),
    ]
    display(pd.DataFrame(rows, columns=["항목", "값"]).astype({"값": str}).set_index("항목"))
    split_of = LABELS["file_name"].map(SPLIT_TABLE.set_index("file_name")["split"])
    class_counts = CLASS_MAP.set_index("category_id")[["dl_name"]].copy()
    class_counts["대회 v1 전체"] = LABELS.groupby("category_id").size()
    class_counts["대회 train"] = LABELS[split_of.eq("train")].groupby("category_id").size()
    class_counts["외부 학습 라벨"] = AIHUB_LABELS.groupby("category_id").size()
    count_columns = ["대회 v1 전체", "대회 train", "외부 학습 라벨"]
    class_counts[count_columns] = class_counts[count_columns].fillna(0).astype(int)
    class_counts["train 합계"] = class_counts["대회 train"] + class_counts["외부 학습 라벨"]
    class_counts["대회+외부 합계"] = class_counts["대회 v1 전체"] + class_counts["외부 학습 라벨"]
    class_counts = class_counts.sort_values("대회 v1 전체")
    class_counts.to_csv(EXPERIMENT_DIR / "aihub_class_counts.csv")
    n_short = int(class_counts["대회+외부 합계"].lt(AIHUB_TARGET_BBOX).sum())
    print(f"대회+외부 합계가 {AIHUB_TARGET_BBOX}개 미만인 클래스: {n_short}/{len(class_counts)}종")
    with pd.option_context("display.max_rows", None):
        display(class_counts)
    ax = class_counts[["대회 train", "외부 학습 라벨"]].plot.barh(
        stacked=True, figsize=(11, 14), color=["steelblue", "darkorange"]
    )
    ax.axvline(AIHUB_TARGET_BBOX, color="red", linestyle="--")
    ax.legend(["competition train", "AI Hub (train only)", f"target {AIHUB_TARGET_BBOX}"])
    ax.set(title="Boxes per class in train", xlabel="boxes", ylabel="category_id")
    plt.tight_layout()
    plt.savefig(EXPERIMENT_DIR / "aihub_class_counts.png", dpi=150)
    plt.show()
    plt.close()
print(f"학습 data.yaml: {TRAIN_DATA_YAML}")
print(f"평가 validation 목록 (대회 사진만): {DATA_YAML.parent / 'val.txt'}")

In [ ]:
# 외부 학습 사진 예시. 초록 = 대회 클래스, 주황 = 외부 전용 새 클래스.
if USE_AIHUB and len(AIHUB_LABELS):
    aihub_image_dir = AIHUB_YOLO_DIR / "images" / "train"
    comp_codes = set(CLASS_MAP["category_id"])
    extra_files = sorted(
        set(AIHUB_LABELS.loc[~AIHUB_LABELS["category_id"].isin(comp_codes), "file_name"])
    )
    preview_pool = extra_files or sorted(set(AIHUB_LABELS["file_name"]))
    preview_files = pd.Series(preview_pool).sample(n=min(4, len(preview_pool)), random_state=SEED)
    fig, axes = plt.subplots(1, len(preview_files), figsize=(4 * len(preview_files), 5.5))
    for axis, file_name in zip(np.atleast_1d(axes), preview_files, strict=True):
        bgr = cv2.imread(str(aihub_image_dir / file_name))
        if bgr is None:
            raise ValueError(f"이미지를 읽을 수 없습니다: {file_name}")
        axis.imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        for box in AIHUB_LABELS[AIHUB_LABELS["file_name"] == file_name].itertuples():
            color = "lime" if box.category_id in comp_codes else "darkorange"
            axis.add_patch(
                Rectangle((box.x, box.y), box.w, box.h, fill=False, edgecolor=color, lw=2)
            )
            axis.text(box.x, box.y - 8, str(box.category_id), color=color, fontsize=8)
        axis.set_title(combo_of(file_name), fontsize=8)
        axis.axis("off")
    fig.suptitle("AI Hub train: lime = competition class, orange = external-only class")
    fig.tight_layout()
    plt.show()
    plt.close(fig)

## 3. 검증 보호·주 지표·조기 중단

mAP75–95는 IoU 0.75, 0.80, 0.85, 0.90, 0.95의 클래스별 AP 평균입니다.
기존 Map7595Trainer로 best.pt와 early stopping 기준을 일치시킵니다.
0점이 이어져도 PATIENCE(10)에폭간 개선이 없으면 중단하도록 stopper를 연결합니다.

혼동행렬의 conf는 명시적으로 적용합니다. 정답 매칭 IoU=0.45는 라이브러리 기본값이며
NMS IoU=0.5와 별도로 기록합니다. Ultralytics Precision/Recall 요약은 IoU=0.5의
max-F1 신뢰도 지점 macro 평균입니다. 고정 conf에서의 혼동행렬 micro P/R은 별도 열입니다.

In [ ]:
class ExactConfusionMatrix(ConfusionMatrix):
    """요청한 신뢰도로 먼저 걸러 혼동행렬의 임계값을 명확하게 적용한다."""

    def process_batch(
        self,
        detections: dict[str, torch.Tensor],
        batch: dict,
        conf: float = 0.25,
        iou_thres: float = CM_MATCH_IOU,
    ) -> None:
        mask = detections["conf"] > conf
        filtered = {key: value[mask] for key, value in detections.items()}
        super().process_batch(filtered, batch, conf=0.0, iou_thres=iou_thres)


class ReadOnlyValidator(DetectionValidator):
    """검증에서 가중치가 바뀌지 않는지 확인하고 평가에 쓴 예측 박스를 보관한다."""

    @torch.inference_mode()
    def __call__(self, trainer=None, model=None, **kwargs) -> dict | None:
        protected = []
        if VERIFY_VALIDATION_WEIGHTS:
            candidates = (
                [trainer.model, trainer.ema.ema]
                if trainer is not None
                else [model]
                if isinstance(model, torch.nn.Module)
                else []
            )
            for candidate in candidates:
                snapshot = {
                    name: tensor.detach().cpu().clone()
                    for name, tensor in candidate.state_dict().items()
                }
                protected.append((candidate, snapshot))
        stats = super().__call__(trainer=trainer, model=model, **kwargs)
        for candidate, snapshot in protected:
            current = candidate.state_dict()
            changed = [
                name
                for name, tensor in snapshot.items()
                if name not in current or not torch.equal(tensor, current[name].detach().cpu())
            ]
            if changed or snapshot.keys() != current.keys():
                raise AssertionError(f"검증 중 가중치/버퍼가 바뀌었습니다: {changed[:5]}")
        return stats

    def init_metrics(self, model: torch.nn.Module) -> None:
        if torch.is_grad_enabled() or model.training:
            raise AssertionError("검증은 gradient 비활성화 + eval 모드여야 합니다.")
        super().init_metrics(model)
        # NMS_ENABLED=True: one2many 헤드 + 클래스별 NMS, False: one2one 헤드(end2end)
        if self.end2end == NMS_ENABLED or self.args.nms is not NMS_ENABLED:
            raise AssertionError("NMS_ENABLED와 검증 후처리 경로(NMS / NMS-free)가 다릅니다.")
        if self.args.agnostic_nms:
            raise AssertionError("클래스별 후처리가 필요합니다(agnostic_nms=False).")
        if self.args.max_det != MAX_DET or self.args.iou != NMS_IOU:
            raise AssertionError("합의한 NMS IoU/max_det와 다릅니다.")
        self.confusion_matrix = ExactConfusionMatrix(names=self.names)
        self.validation_predictions = []

    def postprocess(self, preds: torch.Tensor) -> list[dict[str, torch.Tensor]]:
        outputs = super().postprocess(preds)
        if any(len(output["conf"]) > MAX_DET for output in outputs):
            raise AssertionError("max_det 제한이 적용되지 않았습니다.")
        # 외부 전용 클래스는 대회 클래스가 아니므로 평가·시각화 예측에서 지운다(제출과 같게).
        return [
            {key: value[output["cls"] < N_COMP_CLASSES] for key, value in output.items()}
            for output in outputs
        ]

    def update_metrics(self, preds: list[dict], batch: dict) -> None:
        super().update_metrics(preds, batch)
        if self.training:
            return
        # 별도 predict() 대신 평가한 박스를 원본 이미지 좌표로 되돌려 시각화에 사용한다.
        for index, prediction in enumerate(preds):
            prepared = self._prepare_batch(index, batch)
            original = self.scale_preds(prediction, prepared)
            self.validation_predictions.append(
                {
                    "file_name": Path(prepared["im_file"]).name,
                    "xyxy": original["bboxes"].detach().cpu().tolist(),
                    "yolo_idx": original["cls"].int().cpu().tolist(),
                    "conf": original["conf"].detach().cpu().tolist(),
                }
            )

    def finalize_metrics(self) -> None:
        super().finalize_metrics()
        self.metrics.validation_predictions = self.validation_predictions


class StrictEarlyStopping(EarlyStopping):
    """첫 평가 이후에는 엄격한 점수 증가만 개선으로 인정한다."""

    def __call__(self, epoch: int, fitness: float | None) -> bool:
        if fitness is None:
            return False
        if not np.isfinite(fitness):
            raise ValueError(f"유효하지 않은 검증 점수: {fitness}")
        if self.best_epoch == 0 or fitness > self.best_fitness:
            self.best_epoch, self.best_fitness = epoch, fitness
        delta = epoch - self.best_epoch
        self.possible_stop = delta >= self.patience - 1
        stop = delta >= self.patience
        if stop:
            print(f"Early stopping: {delta}에폭 개선 없음, best epoch={self.best_epoch}")
        return stop


class BaselineTrainer(Map7595Trainer):
    """기존 mAP75–95 학습기에 검증 보호를 연결한다."""

    def get_validator(self) -> ReadOnlyValidator:
        return ReadOnlyValidator(
            self.test_loader,
            save_dir=self.save_dir,
            args=copy(self.args),
            _callbacks=self.callbacks,
        )


def start_training(trainer: BaselineTrainer) -> None:
    """엄격한 stopper를 적용하고 실제 옵티마이저·학습률 그룹을 남긴다.

    이어 학습이면 ultralytics가 last.pt에서 복원한 최고 점수·에폭을 넘겨받아 patience를 이어서 센다.
    """
    restored = trainer.stopper
    trainer.stopper = StrictEarlyStopping(patience=trainer.args.patience)
    trainer.stopper.best_fitness = restored.best_fitness
    trainer.stopper.best_epoch = restored.best_epoch
    if trainer.start_epoch:
        print(
            f"이어 학습: {trainer.start_epoch + 1}에폭부터, "
            f"지금까지 최고 {restored.best_fitness:.5f} (epoch {restored.best_epoch})"
        )
    info = {
        "optimizer": type(trainer.optimizer).__name__,
        "lr0": trainer.args.lr0,
        "group_initial_lrs": [
            float(group.get("initial_lr", group["lr"])) for group in trainer.optimizer.param_groups
        ],
        "trainable_parameters": sum(
            parameter.numel() for parameter in trainer.model.parameters() if parameter.requires_grad
        ),
        "frozen_parameters": sum(
            parameter.numel()
            for parameter in trainer.model.parameters()
            if not parameter.requires_grad
        ),
    }
    write_json(Path(trainer.save_dir) / "optimizer_used.json", info)
    print(json.dumps(info, ensure_ascii=False))


def log_epoch(trainer: BaselineTrainer) -> None:
    """매 에폭 train/val 손실과 검증 지표를 함께 출력한다."""
    if not trainer.validator.training:
        return
    values = {
        "epoch": trainer.epoch + 1,
        **trainer.label_loss_items(trainer.tloss, prefix="train"),
        **{key: float(value) for key, value in trainer.metrics.items()},
    }
    print(json.dumps(values, ensure_ascii=False))

## 4. 단일 holdout 학습과 에폭별 꺾은선 그래프

이 셀에서 실제 학습이 한 번 시작됩니다(SMOKE=True면 동작 확인). 교차검증 반복 학습은 하지 않습니다.
동일한 설정으로 완료한 결과는 재사용하고, 중간에 끊긴 결과(`train/weights/last.pt`가 있음)는
그 체크포인트에서 이어 학습합니다(옵티마이저·EMA·학습률 스케줄·조기 중단 카운트 포함).
끊긴 결과를 덮어쓰지는 않습니다.
OOM이 발생해도 batch=16을 임의로 줄이지 않습니다.

학습 데이터는 2-2절의 `TRAIN_DATA_YAML`(대회 train + 외부 사진, 외부 전용 클래스 포함)이고,
학습 중 검증과 5절 이후 평가는 대회 validation만 씁니다.
외부 사진이 늘어난 만큼 에폭당 시간이 늘어납니다(2-2절의 `train 배수`).

best.pt는 고정한 validation의 mAP75–95가 가장 높은 체크포인트입니다.
학습 뒤 실제 수행된 에폭까지 train/val 손실과 mAP75–95·Precision·Recall을 각각 그립니다.

In [ ]:
def train_baseline() -> Path:
    """단일 holdout으로 사전학습 가중치부터 한 번 학습하고 best.pt 경로를 반환한다.

    끊긴 학습(train/weights/last.pt가 있음)은 그 체크포인트에서 이어 학습한다.
    """
    train_dir = EXPERIMENT_DIR / "train"
    complete_path = train_dir / "completed.json"
    last = train_dir / "weights" / "last.pt"
    if complete_path.is_file():
        complete = json.loads(complete_path.read_text(encoding="utf-8"))
        # 다른 PC·Colab에서 만든 기록이면 경로가 달라도 같은 실험 폴더의 best.pt를 쓴다.
        best = train_dir / "weights" / Path(complete["best_pt"]).name
        if training_config_diff(complete["config"]) or not best.is_file():
            raise ValueError("기존 완료 기록과 설정/가중치가 다릅니다.")
        print(f"완료된 단일 학습 사용: {best}")
        return best

    def mark_complete(best: Path) -> Path:
        if not best.is_file():
            raise FileNotFoundError(f"best.pt가 없습니다: {best}")
        write_json(
            complete_path,
            {
                "best_pt": str(best),
                "config": EXPERIMENT_CONFIG,
                "pretrained_sha256": file_sha256(PRETRAINED_PATH),
            },
        )
        return best

    resume = train_dir.exists()
    if resume and not last.is_file():
        raise FileExistsError(
            f"last.pt 없이 끊긴 결과가 있습니다(첫 에폭 전): {train_dir}. "
            "새 실험 설정 셀부터 실행하세요."
        )
    if resume and is_finished(last):
        print(f"학습은 끝났고 완료 기록만 없습니다. 기록을 남깁니다: {train_dir}")
        return mark_complete(train_dir / "weights" / "best.pt")
    PRETRAINED_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not PRETRAINED_PATH.is_file() and (PROJECT_ROOT / MODEL_NAME).is_file():
        shutil.copy2(PROJECT_ROOT / MODEL_NAME, PRETRAINED_PATH)
    init_seeds(SEED, deterministic=True)
    model = YOLO(str(last if resume else PRETRAINED_PATH))
    model.to(DEVICE)
    model.add_callback("on_train_start", start_training)
    model.add_callback("on_fit_epoch_end", log_epoch)
    args = {
        **AUG_PRESETS[AUG],
        "data": str(TRAIN_DATA_YAML),
        "epochs": TRAIN_EPOCHS,
        "patience": PATIENCE,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "seed": SEED,
        "optimizer": OPTIMIZER,
        "lr0": LR0,
        "freeze": FREEZE,
        "device": DEVICE,
        "workers": WORKERS,
        "deterministic": True,
        "val": True,
        "save": True,
        "plots": True,
        "amp": True,  # MPS에서는 ultralytics가 자동으로 끈다.
        "fraction": TRAIN_FRACTION,
        "cos_lr": False,
        "conf": EVAL_CONF,
        "iou": NMS_IOU,
        "max_det": MAX_DET,
        "nms": NMS_ENABLED,
        "agnostic_nms": False,
        "project": str(train_dir.parent),
        "name": train_dir.name,
        "exist_ok": False,
    }
    if resume:
        # 학습 인자는 last.pt에 저장된 값을 그대로 쓴다. 바뀔 수 있는 데이터 경로·장치·
        # worker 수·결과 폴더만 지금 값으로 넘긴다(ultralytics check_resume이 허용하는 항목).
        print(f"끊긴 학습을 이어 합니다: {last}")
        args = {
            "resume": True,
            "data": str(TRAIN_DATA_YAML),
            "device": DEVICE,
            "workers": WORKERS,
            "save_dir": str(train_dir),
        }
    try:
        model.train(trainer=BaselineTrainer, **args)
        return mark_complete(Path(model.trainer.best))
    finally:
        del model
        gc.collect()
        empty_device_cache()


print("===== combo 단일 holdout | 교차검증 없이 한 번 학습 =====")
BEST_WEIGHTS = train_baseline()
print(f"best.pt: {BEST_WEIGHTS}")

In [ ]:
def plot_epoch_metrics(history: pd.DataFrame, output_path: Path) -> None:
    """에폭별 mAP75–95·Precision·Recall을 각각 꺾은선 그래프로 표시하고 저장한다."""
    columns = [
        (MAIN_METRIC_KEY, "mAP@0.75:0.95", "tab:blue"),
        ("metrics/precision(B)", "Precision (macro, max-F1)", "tab:orange"),
        ("metrics/recall(B)", "Recall (macro, max-F1)", "tab:green"),
    ]
    missing = [key for key, _, _ in columns if key not in history]
    if "epoch" not in history or history.empty or missing:
        raise ValueError(f"에폭별 지표 기록이 부족합니다: {missing}")
    fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)
    for axis, (key, title, color) in zip(axes, columns, strict=True):
        axis.plot(history["epoch"], history[key], marker="o", markersize=3, color=color)
        axis.set(title=title, ylabel="Score", ylim=(0, 1))
        axis.grid(alpha=0.3)
    axes[-1].set_xlabel("Epoch")
    axes[-1].set_xticks(history["epoch"])
    fig.suptitle("Holdout | Validation metrics by epoch")
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.show()
    plt.close(fig)


history = pd.read_csv(EXPERIMENT_DIR / "train" / "results.csv")
history.columns = history.columns.str.strip()
losses = [
    column
    for column in history
    if column.startswith("train/") and f"val/{column.split('/', 1)[1]}" in history
]
if losses:
    fig, axes = plt.subplots(len(losses), 1, figsize=(11, 3 * len(losses)), squeeze=False)
    for axis, train_key in zip(axes[:, 0], losses, strict=True):
        val_key = f"val/{train_key.split('/', 1)[1]}"
        axis.plot(history["epoch"], history[train_key], label=train_key)
        axis.plot(history["epoch"], history[val_key], label=val_key)
        axis.set(xlabel="Epoch", ylabel="Loss")
        axis.grid(alpha=0.3)
        axis.legend()
    fig.suptitle("Holdout | Train / Validation loss")
    fig.tight_layout()
    fig.savefig(EXPERIMENT_DIR / "learning_curves.png", dpi=150)
    plt.show()
    plt.close(fig)
plot_epoch_metrics(history, EXPERIMENT_DIR / "metric_curves.png")

## 5. best.pt 평가 · 두 신뢰도의 혼동행렬 · 클래스별 AP

단일 학습의 같은 best.pt와 고정 validation을 사용하고 conf만 0.001/0.25로 바꿉니다.
두 평가의 원시·정규화 혼동행렬을 저장하고 같은 색상 범위로 비교합니다.
주 성능은 conf=0.001이며 검증에 없는 클래스 AP는 NaN으로 둡니다.
교차검증이나 fold 간 평균·표준편차 계산은 하지 않습니다.
모델 클래스 수에 맞춰 `TRAIN_DATA_YAML`로 평가합니다. validation 목록은 대회 사진 그대로이고,
외부 전용 클래스 예측은 검증기에서 지우므로 혼동행렬은 대회 56종 + background만 남깁니다.


In [ ]:
def per_class_metrics(metrics: object) -> pd.DataFrame:
    """검증에 없는 클래스는 NaN으로 두고 클래스별 AP·학습/검증 수를 만든다."""
    table = CLASS_MAP.copy()
    table["group"] = table["category_id"].map(LABELS["category_id"].value_counts().map(size_group))
    mapping = SPLIT_TABLE.set_index("file_name")["split"]
    box_split = LABELS["file_name"].map(mapping)
    for split in ("train", "val"):
        counts = LABELS.loc[box_split == split, "category_id"].value_counts()
        table[f"n_{split}_bbox"] = table["category_id"].map(counts).fillna(0).astype(int)
    aihub_counts = AIHUB_LABELS["category_id"].value_counts()
    table["n_aihub_bbox"] = table["category_id"].map(aihub_counts).fillna(0).astype(int)
    box = metrics.box
    scores = pd.DataFrame(
        {
            "ap50": box.ap50,
            "ap50_95": box.ap,
            "ap75_95": ap75_95(box.all_ap) if len(box.all_ap) else [],
            "precision_max_f1": box.p,
            "recall_max_f1": box.r,
        },
        index=box.ap_class_index,
    )
    table = table.join(scores, on="yolo_idx")
    present = table["n_val_bbox"] > 0
    table.loc[present, scores.columns] = table.loc[present, scores.columns].fillna(0.0)
    table.loc[table["n_val_bbox"] == 0, scores.columns] = np.nan
    return table


def compare_confusion_matrices(matrices: dict[float, np.ndarray], output_path: Path) -> None:
    """동일한 색상 범위로 두 혼동행렬을 나란히 저장한다."""
    names = [str(value) for value in CLASS_MAP["category_id"]] + ["background"]
    maximum = max(1.0, max(float(matrix.max()) for matrix in matrices.values()))
    fig, axes = plt.subplots(1, len(CM_CONFS), figsize=(24, 11), squeeze=False)
    for axis, confidence in zip(axes[0], CM_CONFS, strict=True):
        mesh = axis.imshow(matrices[confidence], cmap="Blues", vmin=0, vmax=maximum)
        axis.set(
            title=f"conf={confidence:g}, match IoU={CM_MATCH_IOU}",
            xlabel="True",
            ylabel="Predicted",
            xticks=range(len(names)),
            yticks=range(len(names)),
            xticklabels=names,
            yticklabels=names,
        )
        axis.tick_params(axis="x", labelrotation=90, labelsize=6)
        axis.tick_params(axis="y", labelsize=6)
    fig.colorbar(mesh, ax=axes.ravel().tolist(), label="Count", fraction=0.02, pad=0.02)
    fig.savefig(output_path, dpi=170, bbox_inches="tight")
    plt.close(fig)


def evaluate_baseline(weights: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """고정 validation을 두 신뢰도로 평가하고 지표·행렬·실제 평가 예측을 저장한다."""
    out_dir = EXPERIMENT_DIR / "evaluation"
    matrices, rows = {}, []
    main_per_class = None
    class_names = [str(value) for value in CLASS_MAP["category_id"]] + ["background"]
    class_index = dict(zip(CLASS_MAP["category_id"], CLASS_MAP["yolo_idx"], strict=True))
    expected_files = set(SPLIT_TABLE.loc[SPLIT_TABLE["split"] == "val", "file_name"])
    for confidence in CM_CONFS:
        model = YOLO(str(weights))
        try:
            metrics = model.val(
                validator=ReadOnlyValidator,
                data=str(TRAIN_DATA_YAML),
                split="val",
                imgsz=IMGSZ,
                batch=BATCH,
                device=DEVICE,
                workers=WORKERS,
                conf=confidence,
                iou=NMS_IOU,
                max_det=MAX_DET,
                nms=NMS_ENABLED,
                agnostic_nms=False,
                quantize=32,
                plots=True,
                augment=False,
                save_json=False,
                project=str(out_dir),
                name=f"conf_{confidence:g}",
                exist_ok=True,
            )
            predictions = metrics.validation_predictions
            if (
                len(predictions) != len(expected_files)
                or {record["file_name"] for record in predictions} != expected_files
            ):
                raise AssertionError("평가 예측 이미지와 고정 validation이 다릅니다.")
            write_json(out_dir / f"predictions_conf_{confidence:g}.json", predictions)
            full_matrix = metrics.confusion_matrix.matrix
            keep = [*range(N_COMP_CLASSES), full_matrix.shape[0] - 1]  # 대회 클래스 + background
            matrix = full_matrix[np.ix_(keep, keep)].copy()
            if matrix.sum() != full_matrix.sum():
                raise AssertionError("외부 전용 클래스가 혼동행렬에 남았습니다.")
            matrices[confidence] = matrix
            pd.DataFrame(matrix.astype(int), index=class_names, columns=class_names).to_csv(
                out_dir / f"confusion_matrix_conf_{confidence:g}.csv", index_label="predicted/true"
            )
            true_positives = float(np.trace(matrix[:-1, :-1]))
            predicted = float(matrix[:-1, :].sum())
            targets = float(matrix[:, :-1].sum())
            table = per_class_metrics(metrics)
            if confidence == EVAL_CONF:
                main_per_class = table
                table.to_csv(out_dir / "per_class.csv", index=False)
            rows.append(
                {
                    "evaluation": "holdout",
                    "conf": confidence,
                    MAIN_METRIC: map75_95(metrics.box),
                    "precision_macro_max_f1_iou0.5": float(metrics.box.mp),
                    "recall_macro_max_f1_iou0.5": float(metrics.box.mr),
                    "precision_micro_fixed_conf_cm": true_positives / predicted
                    if predicted
                    else 0.0,
                    "recall_micro_fixed_conf_cm": true_positives / targets if targets else 0.0,
                    "cm_match_iou": CM_MATCH_IOU,
                    "n_predictions": int(predicted),
                    "n_ground_truths": int(targets),
                    "classes_not_in_val": int(table["n_val_bbox"].eq(0).sum()),
                    "val_only_classes": int(
                        ((table["n_train_bbox"] == 0) & (table["n_val_bbox"] > 0)).sum()
                    ),
                    **{
                        f"ap75_95_group_{group}": float(
                            table.loc[table["group"] == group, "ap75_95"].mean()
                        )
                        for group in ("21+", "6-20", "≤5")
                    },
                    **{
                        f"confuse_{first}_{second}": int(
                            matrix[class_index[first], class_index[second]]
                            + matrix[class_index[second], class_index[first]]
                        )
                        for first, second in CONFUSION_PAIRS
                        if first in class_index and second in class_index
                    },
                    "aihub_train_images": int(AIHUB_INFO.get("train_images", 0)),
                    "aihub_train_bbox": len(AIHUB_LABELS),
                    "aihub_extra_classes": int(AIHUB_INFO.get("extra_classes", 0)),
                    "validation_batch": BATCH,
                    "val_forward_ms_per_image": float(metrics.speed["inference"]),
                    "nms_iou": NMS_IOU,
                    "max_det": MAX_DET,
                }
            )
        finally:
            del model
            gc.collect()
            empty_device_cache()
    comparison = out_dir / "confusion_matrix_comparison.png"
    compare_confusion_matrices(matrices, comparison)
    result = pd.DataFrame(rows)
    result.to_csv(out_dir / "threshold_comparison.csv", index=False)
    display(result)
    display(Image(filename=str(comparison)))
    return result, main_per_class

In [ ]:
THRESHOLD_RESULTS, PER_CLASS_RESULTS = evaluate_baseline(BEST_WEIGHTS)
THRESHOLD_RESULTS.to_csv(EXPERIMENT_DIR / "threshold_comparison.csv", index=False)
PER_CLASS_RESULTS.to_csv(EXPERIMENT_DIR / "per_class.csv", index=False)
PRIMARY_RESULTS = THRESHOLD_RESULTS.loc[THRESHOLD_RESULTS["conf"] == EVAL_CONF].copy()
metric_columns = [
    MAIN_METRIC,
    "precision_macro_max_f1_iou0.5",
    "recall_macro_max_f1_iou0.5",
    "precision_micro_fixed_conf_cm",
    "recall_micro_fixed_conf_cm",
    "ap75_95_group_21+",
    "ap75_95_group_6-20",
    "ap75_95_group_≤5",
]
SUMMARY = PRIMARY_RESULTS.set_index("evaluation")[metric_columns].T
SUMMARY.to_csv(EXPERIMENT_DIR / "summary_holdout.csv")
display(PRIMARY_RESULTS)
display(SUMMARY)
class_summary = PER_CLASS_RESULTS.set_index(["category_id", "dl_name", "group"])[
    [
        "n_train_bbox",
        "n_aihub_bbox",
        "n_val_bbox",
        "ap75_95",
        "precision_max_f1",
        "recall_max_f1",
    ]
]
class_summary.to_csv(EXPERIMENT_DIR / "per_class_summary_holdout.csv")
display(class_summary)
print("고정 validation 한 개를 평가했습니다. 검증에 없는 클래스 AP는 평가 불가(NaN)입니다.")

## 6. Validation 정답 bbox · 예측 bbox 이미지

conf=0.25 평가에서 실제 사용한 예측을 가져옵니다. 평가를 다시 추론하지 않습니다.
**왼쪽: 정답(초록), 가운데: 예측(주황·점수 표시), 오른쪽: 겹쳐 보기**입니다.
정답은 보정된 v1 라벨, 예측은 원본 이미지 좌표입니다.

고정한 모든 validation 이미지 비교본을 저장합니다. 아래 조회 셀에서 시작 위치를
바꿔 원하는 페이지를 볼 수 있습니다. 표시 신뢰도는 평가 conf=0.001과 별도인 VIS_CONF=0.25입니다.

In [ ]:
def draw_box(axis: object, xyxy: list[float], label: str, color: str, linestyle: str = "-") -> None:
    """원본 픽셀 좌표로 박스와 클래스·점수를 그린다."""
    x1, y1, x2, y2 = xyxy
    axis.add_patch(
        Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            edgecolor=color,
            linewidth=1.8,
            linestyle=linestyle,
        )
    )
    axis.text(
        x1,
        max(0, y1 - 5),
        label,
        color="white",
        fontsize=8,
        bbox={"facecolor": color, "alpha": 0.85, "edgecolor": "none", "pad": 2},
    )


def save_validation_images() -> list[Path]:
    """고정 validation의 모든 이미지에 정답·실제 평가 예측 박스를 그려 저장한다."""
    eval_dir = EXPERIMENT_DIR / "evaluation"
    records = json.loads(
        (eval_dir / f"predictions_conf_{VIS_CONF:g}.json").read_text(encoding="utf-8")
    )
    expected_files = set(SPLIT_TABLE.loc[SPLIT_TABLE["split"] == "val", "file_name"])
    if {record["file_name"] for record in records} != expected_files:
        raise AssertionError("시각화 대상이 validation 목록과 다릅니다.")
    images_by_name = {path.name: path for path in TRAIN_IMAGES}
    category_by_index = CLASS_MAP.set_index("yolo_idx")["category_id"].to_dict()
    image_dir = EXPERIMENT_DIR / "val_images"
    image_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for record in sorted(records, key=lambda item: item["file_name"]):
        file_name = record["file_name"]
        bgr = cv2.imread(str(images_by_name[file_name]))
        if bgr is None:
            raise ValueError(f"이미지를 읽을 수 없습니다: {file_name}")
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        fig, axes = plt.subplots(1, 3, figsize=(15, 6))
        for axis, title in zip(
            axes, ("Ground truth", f"Prediction conf>{VIS_CONF:g}", "Overlay"), strict=True
        ):
            axis.imshow(rgb)
            axis.set_title(title)
            axis.axis("off")
        truth = LABELS.loc[LABELS["file_name"] == file_name]
        for box in truth.itertuples():
            coords = [box.x, box.y, box.x + box.w, box.y + box.h]
            for index in (0, 2):
                draw_box(axes[index], coords, f"GT {box.category_id}", "green")
        for coords, class_index, score in zip(
            record["xyxy"], record["yolo_idx"], record["conf"], strict=True
        ):
            category = category_by_index[class_index]
            for index in (1, 2):
                draw_box(axes[index], coords, f"Pred {category} {score:.3f}", "darkorange", "--")
        if not record["xyxy"]:
            axes[1].text(0.5, 0.05, "No predictions", transform=axes[1].transAxes, ha="center")
        fig.suptitle(f"Holdout | {file_name}", fontsize=9)
        fig.tight_layout()
        path = image_dir / f"{Path(file_name).stem}.jpg"
        fig.savefig(path, dpi=140, bbox_inches="tight")
        plt.close(fig)
        paths.append(path)
    return paths


def show_validation_images(start: int = 0, count: int = 4) -> None:
    """저장된 검증 비교 이미지를 지정한 페이지부터 노트북에 표시한다."""
    paths = sorted((EXPERIMENT_DIR / "val_images").glob("*.jpg"))
    if not paths:
        raise FileNotFoundError("먼저 검증 비교 이미지 저장 셀을 실행하세요.")
    if start < 0 or count < 1 or start >= len(paths):
        raise ValueError(f"start는 0~{len(paths) - 1}, count는 1 이상이어야 합니다.")
    print(f"Holdout: {start + 1}~{min(start + count, len(paths))} / 전체 {len(paths)}장")
    for path in paths[start : start + count]:
        display(Image(filename=str(path)))


paths = save_validation_images()
print(f"GT/예측 비교 이미지 {len(paths)}장 저장")

In [ ]:
# start=0, 4, 8, ...로 바꿔 모든 검증 이미지를 볼 수 있다.
show_validation_images(start=0, count=4)

## 7. 추론속도 · 이미지별 꺾은선 그래프 · 단일 holdout 결과

단일 학습의 best.pt와 validation 이미지로 측정합니다. **batch=1, FP32, 640×640,
conf=0.001, NMS 없음(one2one 헤드), max_det=4**를 적용합니다.
warmup 5회 후 validation 전체를 3회 반복하며 매 측정 전후 장치(CUDA·MPS)를 동기화합니다.

forward_ms는 모델 연산 시간입니다. pipeline_ms는 전처리·GPU 전송·forward·후처리·
Python 호출까지의 시간이며 디스크 읽기는 제외합니다.
학습 배치 16, 학습 중 검증 최대 32, 최종 평가 배치 16과 구분합니다.

각 이미지의 반복 측정 평균으로 forward·전체 추론시간을 꺾은선 그래프로 표시합니다.
각 평균은 같은 이미지의 반복 측정값으로 계산합니다.

In [ ]:
def plot_inference_timings(timings: pd.DataFrame, output_path: Path) -> None:
    """검증 이미지별 반복 측정 평균 추론시간을 꺾은선 그래프로 저장하고 표시한다."""
    required = {"file_name", "forward_ms", "pipeline_ms"}
    if timings.empty or not required <= set(timings.columns):
        raise ValueError("이미지별 추론시간 기록이 없습니다.")
    image_order = timings["file_name"].drop_duplicates().tolist()
    means = timings.groupby("file_name")[["forward_ms", "pipeline_ms"]].mean().reindex(image_order)
    positions = np.arange(1, len(means) + 1)
    fig, axis = plt.subplots(figsize=(12, 5))
    axis.plot(positions, means["forward_ms"], marker=".", label="Model forward")
    axis.plot(positions, means["pipeline_ms"], marker=".", label="Full inference pipeline")
    axis.set(
        title="Holdout | Mean inference time per validation image",
        xlabel="Validation image index",
        ylabel="Time (ms)",
    )
    axis.grid(alpha=0.3)
    axis.legend()
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.show()
    plt.close(fig)


def benchmark_baseline(weights: Path) -> dict:
    """warmup·GPU 동기화 후 batch=1의 모델/전체 추론 시간을 기록한다."""
    val_names = set(SPLIT_TABLE.loc[SPLIT_TABLE["split"] == "val", "file_name"])
    loaded = []
    for path in sorted(TRAIN_IMAGES):
        if path.name in val_names:
            image = cv2.imread(str(path))
            if image is None:
                raise ValueError(f"이미지를 읽을 수 없습니다: {path}")
            loaded.append((path.name, image))
    if not loaded or BENCHMARK_WARMUP < 1 or BENCHMARK_ROUNDS < 1:
        raise ValueError("검증 이미지, warmup, 반복 수는 1 이상이어야 합니다.")
    model = YOLO(str(weights))
    model.to(DEVICE)
    model.model.eval()
    predict_args = {
        "imgsz": IMGSZ,
        "batch": 1,
        "rect": False,
        "device": DEVICE,
        "quantize": 32,
        "conf": EVAL_CONF,
        "iou": NMS_IOU,
        "nms": NMS_ENABLED,
        "agnostic_nms": False,
        "max_det": MAX_DET,
        "augment": False,
        "save": False,
        "verbose": False,
        "stream": False,
    }
    rows = []
    try:
        with torch.inference_mode():
            for _ in range(BENCHMARK_WARMUP):
                model.predict(source=loaded[0][1], **predict_args)
            sync_device()
            for repetition in range(BENCHMARK_ROUNDS):
                for file_name, image in loaded:
                    sync_device()
                    start = perf_counter()
                    prediction = model.predict(source=image, **predict_args)[0]
                    sync_device()
                    elapsed = (perf_counter() - start) * 1000
                    if len(prediction.boxes) > MAX_DET:
                        raise AssertionError("속도 측정 예측의 max_det가 다릅니다.")
                    rows.append(
                        {
                            "evaluation": "holdout",
                            "round": repetition + 1,
                            "file_name": file_name,
                            "pipeline_ms": elapsed,
                            "forward_ms": float(prediction.speed["inference"]),
                            "preprocess_ms": float(prediction.speed["preprocess"]),
                            "postprocess_ms": float(prediction.speed["postprocess"]),
                        }
                    )
    finally:
        del model
        gc.collect()
        empty_device_cache()
    timings = pd.DataFrame(rows)
    summary = {
        "evaluation": "holdout",
        "n_images": len(loaded),
        "n_measurements": len(timings),
        "forward_mean_ms": float(timings["forward_ms"].mean()),
        "pipeline_mean_ms": float(timings["pipeline_ms"].mean()),
        "pipeline_median_ms": float(timings["pipeline_ms"].median()),
        "pipeline_p95_ms": float(timings["pipeline_ms"].quantile(0.95)),
        "forward_fps": 1000 / float(timings["forward_ms"].mean()),
        "pipeline_fps": 1000 / float(timings["pipeline_ms"].mean()),
    }
    out_dir = EXPERIMENT_DIR / "evaluation"
    timings.to_csv(out_dir / "inference_timings.csv", index=False)
    plot_inference_timings(timings, out_dir / "inference_time_curves.png")
    write_json(
        out_dir / "inference_speed.json",
        {
            **summary,
            **ENVIRONMENT,
            "batch": 1,
            "imgsz": IMGSZ,
            "precision": "FP32",
            "conf": EVAL_CONF,
            "nms_iou": NMS_IOU,
            "max_det": MAX_DET,
            "nms": NMS_ENABLED,
            "agnostic_nms": False,
            "disk_io_included": False,
            "warmup": BENCHMARK_WARMUP,
            "rounds": BENCHMARK_ROUNDS,
        },
    )
    return summary


SPEED_RESULTS = pd.DataFrame([benchmark_baseline(BEST_WEIGHTS)])
SPEED_RESULTS.to_csv(EXPERIMENT_DIR / "inference_speed_holdout.csv", index=False)
FINAL_RESULTS = PRIMARY_RESULTS.merge(SPEED_RESULTS, on="evaluation", validate="1:1")
FINAL_RESULTS.to_csv(EXPERIMENT_DIR / "results_holdout.csv", index=False)
speed_columns = ["forward_mean_ms", "pipeline_mean_ms", "forward_fps", "pipeline_fps"]
FINAL_SUMMARY = FINAL_RESULTS.set_index("evaluation")[metric_columns + speed_columns].T
FINAL_SUMMARY.to_csv(EXPERIMENT_DIR / "summary_holdout.csv")
display(FINAL_RESULTS)
display(FINAL_SUMMARY)
print(f"전체 결과: {EXPERIMENT_DIR}")
print("가중치: train/weights/best.pt")
print("혼동행렬: evaluation/conf_0.001 및 conf_0.25")
print("검증 정답/예측 이미지: val_images/")

## 8. 대회 제출 csv (test 842장)

best.pt로 test 이미지 전체를 예측해 대회 제출 형식 csv를 만듭니다. test는 학습·검증에 쓰지 않았습니다.
검증 점수와 같은 조건이 되도록 **conf=0.001, 위에서 정한 후처리(`NMS_ENABLED`, NMS IoU=0.5),
max_det=4, FP32**로 예측합니다. mAP 채점이라 점수 낮은 박스를 남겨도 점수가 떨어지지 않으므로
conf는 평가와 같은 0.001을 씁니다. 외부 전용 클래스 예측은 대회 클래스가 아니므로 지웁니다.

| 열 | 값 |
|---|---|
| annotation_id | 1부터 매긴 행 번호 (image_id 순, 같은 사진 안에서는 score 높은 순) |
| image_id | test 파일 이름의 숫자 (12.png → 12) |
| category_id | 원본 알약 클래스 번호 (YOLO 번호가 아님) |
| bbox_x, bbox_y, bbox_w, bbox_h | 원본 이미지 픽셀 기준 COCO 형식(왼쪽 위 x, y, 너비, 높이) |
| score | 예측 신뢰도 |

결과는 실험 폴더의 `submission/submission_<RUN_TAG>_conf0.001.csv`로 저장합니다.
RUN_TAG(모델·후처리·옵티마이저·lr·epoch·patience·batch·imgsz·증강·max_det)는 설정 셀에서 만들며
결과 폴더 이름에도 들어갑니다.


In [ ]:
SUBMISSION_COLS = [
    "annotation_id",
    "image_id",
    "category_id",
    "bbox_x",
    "bbox_y",
    "bbox_w",
    "bbox_h",
    "score",
]


def make_test_submission(weights: Path) -> tuple[pd.DataFrame, Path]:
    """test 이미지를 검증과 같은 후처리로 예측해 대회 제출 csv를 저장한다."""
    category_by_index = CLASS_MAP.set_index("yolo_idx")["category_id"].to_dict()
    model = YOLO(str(weights))
    rows = []
    n_extra = 0  # 지운 외부 전용 클래스 예측 수
    try:
        # 경로 목록을 한 번에 넘기면 한 배치로 묶여 메모리를 넘으므로 한 장씩 예측한다.
        for path in TEST_IMAGES:
            result = model.predict(
                source=str(path),
                imgsz=IMGSZ,
                device=DEVICE,
                quantize=32,
                conf=EVAL_CONF,
                iou=NMS_IOU,
                nms=NMS_ENABLED,
                agnostic_nms=False,
                max_det=MAX_DET,
                augment=False,
                verbose=False,
            )[0]
            boxes = result.boxes
            if len(boxes) > MAX_DET:
                raise AssertionError("제출 예측의 max_det가 다릅니다.")
            image_id = int(Path(result.path).stem)  # 12.png → 12
            for (x1, y1, x2, y2), class_index, score in zip(
                boxes.xyxy.tolist(), boxes.cls.int().tolist(), boxes.conf.tolist(), strict=True
            ):
                if class_index >= N_COMP_CLASSES:
                    n_extra += 1
                    continue
                rows.append(
                    {
                        "image_id": image_id,
                        "category_id": category_by_index[class_index],
                        "bbox_x": x1,
                        "bbox_y": y1,
                        "bbox_w": x2 - x1,
                        "bbox_h": y2 - y1,
                        "score": score,
                    }
                )
    finally:
        del model
        gc.collect()
        empty_device_cache()
    # image_id 순, 같은 사진 안에서는 score 높은 순으로 정렬하고 annotation_id를 1부터 매긴다.
    submission = pd.DataFrame(rows, columns=SUBMISSION_COLS[1:]).sort_values(
        ["image_id", "score"], ascending=[True, False], ignore_index=True
    )
    submission.insert(0, "annotation_id", submission.index + 1)
    expected_ids = {int(path.stem) for path in TEST_IMAGES}
    missing_ids = expected_ids - set(submission["image_id"])
    if not set(submission["image_id"]) <= expected_ids:
        raise AssertionError("test에 없는 image_id가 있습니다.")
    if submission.isna().any().any() or (submission[["bbox_w", "bbox_h"]] <= 0).any().any():
        raise AssertionError("제출 표에 빈 값이나 크기 0 이하의 박스가 있습니다.")
    if n_extra:
        print(f"외부 전용 클래스 예측 {n_extra}개를 제출에서 뺐습니다.")
    if missing_ids:
        print(f"예측이 하나도 없는 test 이미지 {len(missing_ids)}장: {sorted(missing_ids)[:10]}")
    out_path = EXPERIMENT_DIR / "submission" / f"submission_{RUN_TAG}_conf{EVAL_CONF:g}.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(out_path, index=False, encoding="utf-8")
    return submission, out_path


SUBMISSION, SUBMISSION_PATH = make_test_submission(BEST_WEIGHTS)
display(SUBMISSION.head(10))
per_image = SUBMISSION.groupby("image_id").size()
print(
    f"test {SUBMISSION['image_id'].nunique()}/{len(TEST_IMAGES)}장, 박스 {len(SUBMISSION)}개 "
    f"(사진당 평균 {per_image.mean():.2f}개, score≥{VIS_CONF:g} 박스 "
    f"{int((SUBMISSION['score'] >= VIS_CONF).sum())}개)"
)
print(f"제출 파일: {SUBMISSION_PATH}")

EDA의 그림·캐시·보정 후보와 실제 적용 라벨을 구분합니다. train에만 있는 희귀 클래스의
AP는 검증할 수 없으며, 전체 56종의 일반화 성능을 측정했다고 해석하지 않습니다.
검증에 없는 클래스 수와 클래스별 AP를 단일 holdout 결과에 남깁니다.

참고: [Ultralytics 검증](https://docs.ultralytics.com/modes/val/),
[후처리·신뢰도·정밀도 설정](https://docs.ultralytics.com/usage/cfg/).
실행한 라이브러리 버전과 모든 명시 설정은 experiment.json 및 train/args.yaml에 기록됩니다.